# Label-free shift detection and few-label recalibration for skin lesion classifiers: experiment code

Cells extracted, in the order they were run, from the authors' working Kaggle notebook. The code is unchanged except for this header, the section headings, and one consolidated loop that replaces three run cells (Section 2). Each cell finds its inputs through a small `*_find()` helper that searches `/kaggle/working` and the authors' Kaggle datasets; to rerun elsewhere, copy the files from `results/` into `/kaggle/working` (see README).

**Environment:** Kaggle Python image; GPU (T4) for training, Grad-CAM and corruption logits; CPU for the analysis cells.

## 1. Splits (paper Section 5.2)
Builds `final_splits.csv` from MILK10k (8 retained classes, trunk lesions withheld) plus HAM10000 and PAD-UFES-20. The `near_ood` set (103 lesions of the three excluded classes) is created but not used in the paper.

In [ ]:
import pandas as pd, numpy as np, os
from sklearn.model_selection import train_test_split

s = pd.read_csv('/kaggle/input/datasets/cvaravinda/c3-arc-rcgn-c3/c3/split.csv')
e = pd.read_csv('/kaggle/input/datasets/cvaravinda/c8-arc-rcgn-c8/c8/ext_split.csv')
KEEP = [0,1,3,4,7,8,9,10]

pool = s[(s.domain=='in') & s.label.isin(KEEP)].copy()
tr, rest = train_test_split(pool, test_size=0.30, stratify=pool.label, random_state=42)
cal, te  = train_test_split(rest, test_size=0.50, stratify=rest.label, random_state=42)

def tidy(df, name):
    d = df[['lesion_id','path','label']].copy()
    d['group'] = d['lesion_id']; d['set'] = name
    return d

parts = [tidy(tr,'train'), tidy(cal,'cal'), tidy(te,'test_id')]
trunk = s[s.domain=='ood']
parts.append(tidy(trunk[trunk.label.isin(KEEP)], 'shift_trunk'))
unk = pd.concat([s[(s.domain=='in') & ~s.label.isin(KEEP)],
                 trunk[~trunk.label.isin(KEEP)]])
parts.append(tidy(unk, 'near_ood'))
for dname, tag in [('HAM10000','ext_ham'), ('PAD-UFES-20','ext_pad')]:
    g = e[e.dataset==dname]
    d = g[['image_id','path','label','group']].rename(columns={'image_id':'lesion_id'})
    d['set'] = tag
    parts.append(d)

out = pd.concat(parts, ignore_index=True)
os.makedirs('/kaggle/working', exist_ok=True)
out.to_csv('/kaggle/working/final_splits.csv', index=False)
print(out.groupby('set').agg(n=('path','count'), groups=('group','nunique')))
print('paths ok:', out.groupby('set').path.apply(lambda s: s.head(50).map(os.path.exists).mean()).to_dict())

## 2. EfficientNet-B0 training, exploratory seeds 0-2 (paper Section 5.4)
Augmentation: horizontal and vertical flips, rotation up to 20 degrees, colour jitter 0.1. The checkpoint is chosen by macro-F1 on a stratified 10% hold-out of the training set. In this exploratory phase B0 was also trained with label smoothing (`ls`) and mixup; only cross-entropy (`ce`) and focal loss were carried forward and reported.

In [ ]:
import os, random, warnings, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision, torchvision.transforms as T
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
warnings.filterwarnings('ignore')

OUT = '/kaggle/working'
KEEP = [0,1,3,4,7,8,9,10]; MAP = {l:i for i,l in enumerate(KEEP)}
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev)
df = pd.read_csv(f'{OUT}/final_splits.csv')
df['y'] = df.label.map(MAP).fillna(-1).astype(int)

norm = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
tf_tr = T.Compose([T.Resize((224,224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                   T.RandomRotation(20), T.ColorJitter(0.1,0.1,0.1), T.ToTensor(), norm])
tf_ev = T.Compose([T.Resize((224,224)), T.ToTensor(), norm])

class DS(Dataset):
    def __init__(s, d, tf): s.p, s.y, s.tf = d.path.values, d.y.values, tf
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        return s.tf(Image.open(s.p[i]).convert('RGB')), int(s.y[i])

def loader(d, tf, shuf, bs):
    return DataLoader(DS(d, tf), batch_size=bs, shuffle=shuf, num_workers=2, pin_memory=True)

def loss_fn(z, y, v):
    if v == 'ls': return F.cross_entropy(z, y, label_smoothing=0.1)
    if v == 'focal':
        lp = F.log_softmax(z, 1); lpt = lp.gather(1, y[:,None]).squeeze(1)
        return (-((1 - lpt.exp())**2) * lpt).mean()
    return F.cross_entropy(z, y)

def ece15(p, y, nb=15):
    conf, pred = p.max(1), p.argmax(1); acc = (pred == y).astype(float)
    e, b = 0.0, np.linspace(0, 1, nb+1)
    for lo, hi in zip(b[:-1], b[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e

@torch.no_grad()
def predict(model, d):
    model.eval(); out = []
    for x, _ in loader(d, tf_ev, False, 64):
        with torch.cuda.amp.autocast():
            out.append(model(x.to(dev)).float().cpu())
    return torch.cat(out).numpy()

WD = '/kaggle/input/models/cvaravinda/efficientnet-b0-rwightman-7f5810bc/tensorflow2/default/1'

def run(variant, seed, epochs=15):
    tag = f'{variant}_s{seed}'; f = f'{OUT}/preds_{tag}.npz'
    if os.path.exists(f): print('already done:', tag); return
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    tr = df[df.set=='train']
    tr, vs = train_test_split(tr, test_size=0.10, stratify=tr.y, random_state=42)
    W = os.path.join(WD, [x for x in os.listdir(WD) if x.endswith(('.pth', '.pt'))][0])
    model = torchvision.models.efficientnet_b0(weights=None)
    model.load_state_dict(torch.load(W, map_location='cpu'))
    model.classifier[1] = nn.Linear(1280, len(KEEP)); model.to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    scaler = torch.cuda.amp.GradScaler(); best, best_sd = -1, None
    for ep in range(epochs):
        model.train()
        for x, y in loader(tr, tf_tr, True, 32):
            x, y = x.to(dev), y.to(dev)
            with torch.cuda.amp.autocast():
                if variant == 'mixup':
                    lam = np.random.beta(0.2, 0.2); idx = torch.randperm(x.size(0), device=dev)
                    z = model(lam*x + (1-lam)*x[idx])
                    loss = lam*F.cross_entropy(z, y) + (1-lam)*F.cross_entropy(z, y[idx])
                else:
                    loss = loss_fn(model(x), y, variant)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sch.step()
        zv = predict(model, vs); mf = f1_score(vs.y, zv.argmax(1), average='macro')
        print(f'{tag} ep{ep+1} val_sel macroF1={mf:.3f}')
        if mf > best: best, best_sd = mf, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_sd)
    res = {}
    for st in df.set.unique():
        d = df[df.set == st]; res[f'z_{st}'] = predict(model, d); res[f'y_{st}'] = d.y.values
    np.savez_compressed(f, **res); torch.save(best_sd, f'{OUT}/model_{tag}.pt')
    z, y = res['z_test_id'], res['y_test_id']; p = torch.softmax(torch.tensor(z), 1).numpy()
    print(f'{tag} TEST_ID acc={(z.argmax(1)==y).mean():.3f} macroF1={f1_score(y, z.argmax(1), average="macro"):.3f} ECE15={ece15(p, y):.3f}')

print('ready')

In [ ]:
# Original runs: run('ce', 0) in an earlier, otherwise identical version of the cell above;
# then run(v, 0) for v in ['ls', 'focal', 'mixup']; then all four variants for seeds 1 and 2.
# run() skips any model whose prediction file already exists.
for seed in [0, 1, 2]:
    for v in ['ce', 'ls', 'focal', 'mixup']:
        run(v, seed)

In [ ]:
import shutil, glob, os
os.chdir('/kaggle/working')
shutil.rmtree('calib_results', ignore_errors=True); os.makedirs('calib_results')
for f in glob.glob('preds_*.npz') + glob.glob('model_*.pt') + ['final_splits.csv']:
    shutil.copy(f, 'calib_results/')
shutil.make_archive('calib_results_all', 'zip', 'calib_results')
print(sorted(os.listdir('calib_results')))
print('zip MB:', round(os.path.getsize('calib_results_all.zip')/1e6, 1))

## 3. EfficientNet-B3 training, exploratory seeds 0-2

In [ ]:
# ---- Cell B1: EfficientNet-B3 second backbone, CE + focal, seeds 0-2 (resumable) ----
import os, time, shutil, random, warnings, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision, torchvision.transforms as T
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
warnings.filterwarnings('ignore')

b3_OUT = '/kaggle/working'
b3_W = '/kaggle/input/models/anandkoirala/efficientnet-b3-rwightman-b3899882-pth/pytorch/default/1/efficientnet_b3_rwightman-b3899882.pth'
b3_KEEP = [0,1,3,4,7,8,9,10]; b3_MAP = {l: i for i, l in enumerate(b3_KEEP)}
b3_dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', b3_dev); assert os.path.exists(b3_W), 'B3 weights not attached'
if not os.path.exists(f'{b3_OUT}/final_splits.csv'):
    shutil.copy('/kaggle/input/datasets/cvaravinda/calib-results-all2/final_splits.csv', f'{b3_OUT}/final_splits.csv')
b3_df = pd.read_csv(f'{b3_OUT}/final_splits.csv')
b3_df['y'] = b3_df.label.map(b3_MAP).fillna(-1).astype(int)

b3_norm = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
b3_tf_tr = T.Compose([T.Resize((224,224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                      T.RandomRotation(20), T.ColorJitter(0.1,0.1,0.1), T.ToTensor(), b3_norm])
b3_tf_ev = T.Compose([T.Resize((224,224)), T.ToTensor(), b3_norm])

class B3DS(Dataset):
    def __init__(s, d, tf): s.p, s.y, s.tf = d.path.values, d.y.values, tf
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return s.tf(Image.open(s.p[i]).convert('RGB')), int(s.y[i])

def b3_loader(d, tf, shuf, bs):
    return DataLoader(B3DS(d, tf), batch_size=bs, shuffle=shuf, num_workers=2, pin_memory=True)

def b3_loss(z, y, v):
    if v == 'focal':
        lp = F.log_softmax(z, 1); lpt = lp.gather(1, y[:, None]).squeeze(1)
        return (-((1 - lpt.exp()) ** 2) * lpt).mean()
    return F.cross_entropy(z, y)

def b3_ece(p, y, nb=15):
    conf, pred = p.max(1), p.argmax(1); acc = (pred == y).astype(float); e = 0.0
    b = np.linspace(0, 1, nb + 1)
    for lo, hi in zip(b[:-1], b[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e

@torch.no_grad()
def b3_predict(model, d):
    model.eval(); out = []
    for x, _ in b3_loader(d, b3_tf_ev, False, 64):
        with torch.cuda.amp.autocast(): out.append(model(x.to(b3_dev)).float().cpu())
    return torch.cat(out).numpy()

def b3_run(variant, seed, epochs=15):
    tag = f'b3_{variant}_s{seed}'; f = f'{b3_OUT}/preds_{tag}.npz'
    if os.path.exists(f): print('already done:', tag); return
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    tr = b3_df[b3_df.set == 'train']
    tr, vs = train_test_split(tr, test_size=0.10, stratify=tr.y, random_state=42)
    model = torchvision.models.efficientnet_b3(weights=None)
    model.load_state_dict(torch.load(b3_W, map_location='cpu'))
    model.classifier[1] = nn.Linear(1536, len(b3_KEEP)); model.to(b3_dev)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    scaler = torch.cuda.amp.GradScaler(); best, best_sd = -1, None; t0 = time.time()
    for ep in range(epochs):
        model.train()
        for x, y in b3_loader(tr, b3_tf_tr, True, 32):
            x, y = x.to(b3_dev), y.to(b3_dev)
            with torch.cuda.amp.autocast(): loss = b3_loss(model(x), y, variant)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sch.step()
        mf = f1_score(vs.y, b3_predict(model, vs).argmax(1), average='macro')
        if ep == 0 or (ep + 1) % 5 == 0: print(f'{tag} ep{ep+1} val_sel macroF1={mf:.3f} ({round(time.time()-t0)}s)')
        if mf > best: best, best_sd = mf, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_sd); res = {}
    for st in b3_df.set.unique():
        d = b3_df[b3_df.set == st]; res[f'z_{st}'] = b3_predict(model, d); res[f'y_{st}'] = d.y.values
    np.savez_compressed(f, **res); torch.save(best_sd, f'{b3_OUT}/model_{tag}.pt')
    z, y = res['z_test_id'], res['y_test_id']; p = torch.softmax(torch.tensor(z), 1).numpy()
    print(f'{tag} TEST_ID acc={(z.argmax(1)==y).mean():.3f} macroF1={f1_score(y, z.argmax(1), average="macro"):.3f} ECE15={b3_ece(p, y):.3f}')

b3_T0 = time.time()
for b3_seed in [0, 1, 2]:
    for b3_v in ['ce', 'focal']:
        if time.time() - b3_T0 > 170 * 60: print('time limit reached: run this cell again to resume'); break
        b3_run(b3_v, b3_seed)
print('\nB3 pred files done:', len([x for x in os.listdir(b3_OUT) if x.startswith('preds_b3_')]), 'of 6')

In [ ]:
# ---- Cell B2: zip + sanity-check the B3 results ----
import glob, os, zipfile, numpy as np
b2_p = sorted(glob.glob('/kaggle/working/preds_b3_*.npz'))
b2_m = sorted(glob.glob('/kaggle/working/model_b3_*.pt'))
print('preds:', len(b2_p), '(expected 6) | models:', len(b2_m), '(expected 6)')
for f in b2_p:
    z = np.load(f)
    print(os.path.basename(f), 'keys:', len(z.files), '(expected 14) | test_id logits:', z['z_test_id'].shape,
          '| ext_pad:', z['z_ext_pad'].shape, '| ext_ham:', z['z_ext_ham'].shape)
out = '/kaggle/working/calib_results_b3.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in b2_p + b2_m: zf.write(f, os.path.basename(f))
print('zip size MB:', round(os.path.getsize(out) / 1e6, 1))

## 4. Exploratory calibration ladder and label-free matcher (seeds 0-2)
L3: raw, clean-fit, ensemble and target-fit ECE. C1: logits on the corrupted calibration set (7 kinds). M1-M2: matched versus clean-fit scaling and sign tests. After this section the hypotheses and decision rules for the matcher were written down (Section 6).

In [ ]:
# ---- Cell L3: calibration ladder, B0 vs B3 (CPU, read-only on preds) ----
import os, numpy as np, pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import softmax, log_softmax

def l3_find(name):
    for d in ['/kaggle/working', '/kaggle/input/datasets/cvaravinda/calib-results-b3', '/kaggle/input/datasets/cvaravinda/calib-results-all2']:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)

def l3_ece(p, y, nb=15):
    conf, pred = p.max(1), p.argmax(1); acc = (pred == y).astype(float); e = 0.0
    b = np.linspace(0, 1, nb + 1)
    for lo, hi in zip(b[:-1], b[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e
def l3_nll(p, y): return float(-np.mean(np.log(p[np.arange(len(y)), y] + 1e-12)))
def l3_brier(p, y): oh = np.eye(p.shape[1])[y]; return float(np.mean(((p - oh) ** 2).sum(1)))
def l3_fitT(z, y):
    f = lambda t: -np.mean(log_softmax(z / t, 1)[np.arange(len(y)), y])
    return minimize_scalar(f, bounds=(0.05, 10), method='bounded').x
def l3_met(p, y): return dict(acc=float((p.argmax(1) == y).mean()), ece=l3_ece(p, y), brier=l3_brier(p, y), nll=l3_nll(p, y))

SETS = ['test_id', 'shift_trunk', 'ext_ham', 'ext_pad']
BB = [('B0', ''), ('B3', 'b3_')]; VARS = ['ce', 'focal']; SEEDS = [0, 1, 2]
rows, ens = [], []
rng = np.random.RandomState(0)
for bb, pre in BB:
    for v in VARS:
        D = {s: np.load(l3_find(f'preds_{pre}{v}_s{s}.npz')) for s in SEEDS}
        for s in SEEDS:
            d = D[s]; zc, yc = d['z_cal'], d['y_cal']; T = l3_fitT(zc, yc)
            for st in SETS:
                z, y = d[f'z_{st}'], d[f'y_{st}']
                rows.append(dict(bb=bb, v=v, seed=s, set=st, method='raw', **l3_met(softmax(z, 1), y)))
                rows.append(dict(bb=bb, v=v, seed=s, set=st, method='cleanTS', **l3_met(softmax(z / T, 1), y)))
                if st in ('ext_ham', 'ext_pad'):
                    acc = []
                    for r in range(20):
                        idx = rng.permutation(len(y)); a, b = idx[:100], idx[100:]
                        To = l3_fitT(z[a], y[a]); acc.append(l3_met(softmax(z[b] / To, 1), y[b]))
                    rows.append(dict(bb=bb, v=v, seed=s, set=st, method='oracle100', **{k: float(np.mean([x[k] for x in acc])) for k in acc[0]}))
        # ensemble over seeds, T refit on the ensemble
        pc = np.mean([softmax(D[s]['z_cal'], 1) for s in SEEDS], 0); Te = l3_fitT(np.log(pc + 1e-12), D[0]['y_cal'])
        for st in SETS:
            y = D[0][f'y_{st}']; pe = np.mean([softmax(D[s][f'z_{st}'], 1) for s in SEEDS], 0)
            ens.append(dict(bb=bb, v=v, set=st, method='ens_raw', **l3_met(pe, y)))
            ens.append(dict(bb=bb, v=v, set=st, method='ens_TS', **l3_met(softmax(np.log(pe + 1e-12) / Te, 1), y)))

R = pd.DataFrame(rows); E = pd.DataFrame(ens)
R.to_csv('/kaggle/working/l3_rows.csv', index=False); E.to_csv('/kaggle/working/l3_ens.csv', index=False)
def l3_show(metric):
    g = R.groupby(['bb', 'v', 'method', 'set'])[metric].agg(['mean', 'std']).reset_index()
    g['txt'] = g['mean'].map('{:.3f}'.format) + '±' + g['std'].map('{:.3f}'.format)
    t = g.pivot_table(index=['bb', 'v', 'method'], columns='set', values='txt', aggfunc='first')[SETS]
    e = E.pivot_table(index=['bb', 'v', 'method'], columns='set', values=metric)[SETS].round(3).astype(str)
    print(f'\n=== {metric.upper()} (mean±sd over 3 seeds; ensemble rows have no sd) ===')
    print(pd.concat([t, e]).sort_index().to_string())
for m in ['acc', 'ece', 'nll']: l3_show(m)

In [ ]:
# ---- Cell C1: logits of B0 and B3 (CE, focal; seeds 0-2) on corrupted calibration images ----
import os, io, numpy as np, pandas as pd, torch, torch.nn as nn, torchvision, torchvision.transforms as T
from PIL import Image, ImageFilter
c1_dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', c1_dev)
def c1_find(name):
    for d in ['/kaggle/working', '/kaggle/input/datasets/cvaravinda/calib-results-b3', '/kaggle/input/datasets/cvaravinda/calib-results-all2']:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
c1_df = pd.read_csv(c1_find('final_splits.csv')); c1_cal = c1_df[c1_df.set == 'cal']
# c1_base = [Image.open(p).convert('RGB').resize((224, 224)) for p in c1_cal.path]
c1_base = [T.Resize((224, 224))(Image.open(p).convert('RGB')) for p in c1_cal.path]
C1_KINDS = ['clean', 'blur', 'jpeg', 'color', 'noise', 'lowres', 'combo']
c1_cj = T.ColorJitter(0.3, 0.3, 0.3, 0.05)

def c1_corrupt(img, kind, rng):
    def blur(im): return im.filter(ImageFilter.GaussianBlur(rng.uniform(1, 2.5)))
    def jpeg(im):
        b = io.BytesIO(); im.save(b, 'JPEG', quality=int(rng.uniform(35, 70))); b.seek(0); return Image.open(b).convert('RGB')
    def color(im): torch.manual_seed(int(rng.randint(1_000_000_000))); return c1_cj(im)
    def noise(im):
        a = np.asarray(im).astype(np.float32) + rng.normal(0, rng.uniform(4, 10), (224, 224, 3))
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    def lowres(im): return im.resize((96, 96), Image.BILINEAR).resize((224, 224), Image.BILINEAR)
    if kind == 'clean': return img
    if kind == 'combo': return lowres(color(jpeg(blur(img))))
    return dict(blur=blur, jpeg=jpeg, color=color, noise=noise, lowres=lowres)[kind](img)

c1_arr = {}
for ki, k in enumerate(C1_KINDS):
    c1_arr[k] = np.stack([np.asarray(c1_corrupt(im, k, np.random.RandomState(10000 * ki + i))) for i, im in enumerate(c1_base)])
print('corrupted arrays:', {k: v.shape for k, v in c1_arr.items()})
c1_mu = torch.tensor([0.485, 0.456, 0.406], device=c1_dev).view(1, 3, 1, 1)
c1_sd = torch.tensor([0.229, 0.224, 0.225], device=c1_dev).view(1, 3, 1, 1)

def c1_model(bb, path):
    if bb == 'B0': m = torchvision.models.efficientnet_b0(weights=None); m.classifier[1] = nn.Linear(1280, 8)
    else:          m = torchvision.models.efficientnet_b3(weights=None); m.classifier[1] = nn.Linear(1536, 8)
    m.load_state_dict(torch.load(path, map_location='cpu')); return m.to(c1_dev).eval()

@torch.no_grad()
def c1_logits(m, arr):
    out = []
    for i in range(0, len(arr), 64):
        x = torch.from_numpy(arr[i:i + 64]).to(c1_dev).permute(0, 3, 1, 2).float().div(255)
        with torch.cuda.amp.autocast(): out.append(m((x - c1_mu) / c1_sd).float().cpu())
    return torch.cat(out).numpy()

c1_res = {}; c1_maxdiff = 0
for bb, pre in [('B0', ''), ('B3', 'b3_')]:
    for v in ['ce', 'focal']:
        for s in [0, 1, 2]:
            m = c1_model(bb, c1_find(f'model_{pre}{v}_s{s}.pt'))
            for k in C1_KINDS: c1_res[f'{bb}_{v}_s{s}__{k}'] = c1_logits(m, c1_arr[k])
            ref = np.load(c1_find(f'preds_{pre}{v}_s{s}.npz'))['z_cal']
            c1_maxdiff = max(c1_maxdiff, float(np.abs(c1_res[f'{bb}_{v}_s{s}__clean'] - ref).max()))
            del m; torch.cuda.empty_cache()
        print('done', bb, v)
np.savez_compressed('/kaggle/working/corr_cal_logits_v2.npz', **c1_res)
print('keys:', len(c1_res), '(expected 84) | shape:', c1_res['B0_ce_s0__blur'].shape, '(expected (487, 8))')
print('max |clean logits - saved z_cal|:', round(c1_maxdiff, 4), '(should be tiny, under about 0.05)')
print('file MB:', round(os.path.getsize('/kaggle/working/corr_cal_logits_v2.npz') / 1e6, 1))

In [ ]:
# ---- Cell M1: confidence-matched scaling vs clean scaling, bootstrap CIs (CPU) ----
import os, numpy as np, pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import softmax, log_softmax
from scipy.stats import wasserstein_distance

def m1_find(name):
    for d in ['/kaggle/working', '/kaggle/input/datasets/cvaravinda/calib-results-b3', '/kaggle/input/datasets/cvaravinda/calib-results-all2']:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
def m1_fitT(z, y):
    f = lambda t: -np.mean(log_softmax(z / t, 1)[np.arange(len(y)), y])
    return minimize_scalar(f, bounds=(0.05, 10), method='bounded').x
def m1_wece(conf, acc, w, nb=15):
    b = (np.ceil(conf * nb) - 1).clip(0, nb - 1).astype(int)
    sw = np.bincount(b, weights=w, minlength=nb); sa = np.bincount(b, weights=w * acc, minlength=nb); sc = np.bincount(b, weights=w * conf, minlength=nb)
    m = sw > 0
    return float((sw[m] / w.sum() * np.abs(sa[m] / sw[m] - sc[m] / sw[m])).sum())

SETS = ['test_id', 'shift_trunk', 'ext_ham', 'ext_pad']
KINDS = ['clean', 'blur', 'jpeg', 'color', 'noise', 'lowres', 'combo']
CFG = [('B0', '', 'ce'), ('B0', '', 'focal'), ('B3', 'b3_', 'ce'), ('B3', 'b3_', 'focal')]
m1_df = pd.read_csv(m1_find('final_splits.csv'))
m1_gid = {st: pd.factorize(m1_df[m1_df.set == st].group.astype(str))[0] for st in SETS}
CZ = np.load('/kaggle/working/corr_cal_logits_v2.npz')
store, rows = {}, []
for bb, pre, v in CFG:
    for s in [0, 1, 2]:
        d = np.load(m1_find(f'preds_{pre}{v}_s{s}.npz')); yc = d['y_cal']
        Zc = {k: CZ[f'{bb}_{v}_s{s}__{k}'] for k in KINDS}
        Tk = {k: m1_fitT(Zc[k], yc) for k in KINDS}
        mp = {k: softmax(Zc[k], 1).max(1) for k in KINDS}
        for st in SETS:
            z, y = d[f'z_{st}'], d[f'y_{st}']; pr = softmax(z, 1); acc = (pr.argmax(1) == y).astype(float)
            dist = {k: wasserstein_distance(pr.max(1), mp[k]) for k in KINDS}; pick = min(dist, key=dist.get)
            cf = {'raw': pr.max(1)}
            for k in KINDS: cf[f'TS_{k}'] = softmax(z / Tk[k], 1).max(1)
            cf['cleanTS'] = cf['TS_clean']; cf['matched'] = cf[f'TS_{pick}']
            store[(bb, v, s, st)] = dict(acc=acc, cf=cf, pick=pick)
            one = np.ones(len(acc))
            for m, c in cf.items(): rows.append(dict(bb=bb, v=v, seed=s, set=st, method=m, ece=m1_wece(c, acc, one), pick=pick))
R = pd.DataFrame(rows); R.to_csv('/kaggle/working/m1_rows.csv', index=False)

print('=== ECE, mean±sd over 3 seeds ===')
t = R[R.method.isin(['raw', 'cleanTS', 'matched'])].groupby(['bb', 'v', 'method', 'set']).ece.agg(['mean', 'std']).reset_index()
t['txt'] = t['mean'].map('{:.3f}'.format) + '±' + t['std'].map('{:.3f}'.format)
print(t.pivot_table(index=['bb', 'v', 'method'], columns='set', values='txt', aggfunc='first')[SETS].to_string())

print('\n=== matcher picks (count over 3 seeds) ===')
for bb, pre, v in CFG:
    for st in SETS:
        print(f'{bb} {v:5s} {st:11s}', pd.Series([store[(bb, v, s, st)]['pick'] for s in [0, 1, 2]]).value_counts().to_dict())

print('\n=== paired group bootstrap: dECE = matched - cleanTS (negative = matched better), 1000 resamples ===')
rng = np.random.RandomState(0)
for bb, pre, v in CFG:
    for st in SETS:
        gid = m1_gid[st]; G = gid.max() + 1; dl = []
        for r in range(1000):
            w = np.bincount(rng.randint(0, G, G), minlength=G)[gid].astype(float)
            dl.append(np.mean([m1_wece(store[(bb, v, s, st)]['cf']['matched'], store[(bb, v, s, st)]['acc'], w) -
                               m1_wece(store[(bb, v, s, st)]['cf']['cleanTS'], store[(bb, v, s, st)]['acc'], w) for s in [0, 1, 2]]))
        lo, hi = np.percentile(dl, [2.5, 97.5])
        print(f'{bb} {v:5s} {st:11s} dECE={np.mean(dl):+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  {"<- excludes 0" if (lo > 0 or hi < 0) else ""}')

In [ ]:
# ---- Cell M2: per-seed matched-vs-clean ECE, sign test, gap closed vs oracle ----
import numpy as np, pandas as pd
from scipy.stats import binomtest
R = pd.read_csv('/kaggle/working/m1_rows.csv'); L = pd.read_csv('/kaggle/working/l3_rows.csv')
orc = L[L.method == 'oracle100'].set_index(['bb', 'v', 'seed', 'set']).ece
P = R[R.method.isin(['cleanTS', 'matched'])].pivot_table(index=['bb', 'v', 'seed', 'set'], columns='method', values='ece').reset_index()
P = P.merge(R[R.method == 'matched'][['bb', 'v', 'seed', 'set', 'pick']], on=['bb', 'v', 'seed', 'set'])
P['delta'] = P.matched - P.cleanTS
P['oracle'] = [orc.get((b, v, s, st), np.nan) for b, v, s, st in zip(P.bb, P.v, P.seed, P.set)]
P['gap_closed'] = (P.cleanTS - P.matched) / (P.cleanTS - P.oracle)
for st in ['ext_pad', 'ext_ham']:
    d = P[P.set == st]
    print(f'\n=== {st} ===')
    print(d[['bb', 'v', 'seed', 'cleanTS', 'matched', 'oracle', 'delta', 'gap_closed', 'pick']].round(3).to_string(index=False))
    k = int((d.delta < 0).sum())
    print(f'improved in {k}/12 seed-configs | mean delta {d.delta.mean():+.3f} | sign-test p={binomtest(k, 12, 0.5).pvalue:.3f}')
    print('mean gap closed (where oracle < cleanTS):', round(d[d.cleanTS > d.oracle].gap_closed.mean(), 2))

## 5. Confirmatory models: fresh seeds 3-5, B0 and B3, CE and focal

In [ ]:
# ---- Cell G1: FRESH seeds 3,4,5 for B0 and B3 (CE + focal) - confirmatory run, resumable ----
import os, time, shutil, random, warnings, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision, torchvision.transforms as T
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
warnings.filterwarnings('ignore')

g1_OUT = '/kaggle/working'
g1_B0DIR = '/kaggle/input/models/cvaravinda/efficientnet-b0-rwightman-7f5810bc/tensorflow2/default/1'
g1_B3W = '/kaggle/input/models/anandkoirala/efficientnet-b3-rwightman-b3899882-pth/pytorch/default/1/efficientnet_b3_rwightman-b3899882.pth'
g1_B0W = os.path.join(g1_B0DIR, [x for x in os.listdir(g1_B0DIR) if x.endswith(('.pth', '.pt'))][0])
assert os.path.exists(g1_B3W), 'B3 weights not attached'
g1_dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', g1_dev)
if not os.path.exists(f'{g1_OUT}/final_splits.csv'):
    for d in ['calib-results-derived', 'calib-results-all2']:
        p = f'/kaggle/input/datasets/cvaravinda/{d}/final_splits.csv'
        if os.path.exists(p): shutil.copy(p, f'{g1_OUT}/final_splits.csv'); break
g1_KEEP = [0, 1, 3, 4, 7, 8, 9, 10]; g1_MAP = {l: i for i, l in enumerate(g1_KEEP)}
g1_df = pd.read_csv(f'{g1_OUT}/final_splits.csv'); g1_df['y'] = g1_df.label.map(g1_MAP).fillna(-1).astype(int)
g1_norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
g1_tf_tr = T.Compose([T.Resize((224, 224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
                      T.ColorJitter(0.1, 0.1, 0.1), T.ToTensor(), g1_norm])
g1_tf_ev = T.Compose([T.Resize((224, 224)), T.ToTensor(), g1_norm])

class G1DS(Dataset):
    def __init__(s, d, tf): s.p, s.y, s.tf = d.path.values, d.y.values, tf
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return s.tf(Image.open(s.p[i]).convert('RGB')), int(s.y[i])
def g1_loader(d, tf, shuf, bs): return DataLoader(G1DS(d, tf), batch_size=bs, shuffle=shuf, num_workers=2, pin_memory=True)
def g1_loss(z, y, v):
    if v == 'focal':
        lp = F.log_softmax(z, 1); lpt = lp.gather(1, y[:, None]).squeeze(1); return (-((1 - lpt.exp()) ** 2) * lpt).mean()
    return F.cross_entropy(z, y)
def g1_ece(p, y, nb=15):
    conf, pred = p.max(1), p.argmax(1); acc = (pred == y).astype(float); e = 0.0; b = np.linspace(0, 1, nb + 1)
    for lo, hi in zip(b[:-1], b[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e
@torch.no_grad()
def g1_predict(model, d):
    model.eval(); out = []
    for x, _ in g1_loader(d, g1_tf_ev, False, 64):
        with torch.cuda.amp.autocast(): out.append(model(x.to(g1_dev)).float().cpu())
    return torch.cat(out).numpy()

def g1_build(bb):
    if bb == 'B0':
        m = torchvision.models.efficientnet_b0(weights=None); m.load_state_dict(torch.load(g1_B0W, map_location='cpu')); m.classifier[1] = nn.Linear(1280, 8)
    else:
        m = torchvision.models.efficientnet_b3(weights=None); m.load_state_dict(torch.load(g1_B3W, map_location='cpu')); m.classifier[1] = nn.Linear(1536, 8)
    return m.to(g1_dev)

def g1_run(bb, variant, seed, epochs=15):
    pre = '' if bb == 'B0' else 'b3_'; tag = f'{pre}{variant}_s{seed}'; f = f'{g1_OUT}/preds_{tag}.npz'
    if os.path.exists(f): print('already done:', tag); return
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    tr = g1_df[g1_df.set == 'train']; tr, vs = train_test_split(tr, test_size=0.10, stratify=tr.y, random_state=42)
    model = g1_build(bb); opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs); scaler = torch.cuda.amp.GradScaler(); best, best_sd = -1, None; t0 = time.time()
    for ep in range(epochs):
        model.train()
        for x, y in g1_loader(tr, g1_tf_tr, True, 32):
            x, y = x.to(g1_dev), y.to(g1_dev)
            with torch.cuda.amp.autocast(): loss = g1_loss(model(x), y, variant)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sch.step(); mf = f1_score(vs.y, g1_predict(model, vs).argmax(1), average='macro')
        if ep == 0 or (ep + 1) % 5 == 0: print(f'{tag} ep{ep+1} val_sel macroF1={mf:.3f} ({round(time.time()-t0)}s)')
        if mf > best: best, best_sd = mf, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_sd); res = {}
    for st in g1_df.set.unique():
        d = g1_df[g1_df.set == st]; res[f'z_{st}'] = g1_predict(model, d); res[f'y_{st}'] = d.y.values
    np.savez_compressed(f, **res); torch.save(best_sd, f'{g1_OUT}/model_{tag}.pt')
    z, y = res['z_test_id'], res['y_test_id']; p = torch.softmax(torch.tensor(z), 1).numpy()
    print(f'{tag} TEST_ID acc={(z.argmax(1)==y).mean():.3f} macroF1={f1_score(y, z.argmax(1), average="macro"):.3f} ECE15={g1_ece(p, y):.3f}')

g1_T0 = time.time(); g1_stop = False
for g1_seed in [3, 4, 5]:
    for g1_bb in ['B0', 'B3']:
        for g1_v in ['ce', 'focal']:
            if time.time() - g1_T0 > 170 * 60: g1_stop = True; break
            g1_run(g1_bb, g1_v, g1_seed)
        if g1_stop: break
    if g1_stop: print('time limit reached: run this cell again to resume'); break
n_done = len([f for f in os.listdir(g1_OUT) if f.startswith('preds_') and any(f.endswith(f'_s{s}.npz') for s in [3, 4, 5])])
print('\nfresh-seed pred files done:', n_done, 'of 12')

In [ ]:
# ---- Cell G2: zip + sanity-check the fresh-seed results ----
import glob, os, zipfile, numpy as np
g2_p = sorted(f for f in glob.glob('/kaggle/working/preds_*.npz') if any(f.endswith(f'_s{s}.npz') for s in [3, 4, 5]))
g2_m = sorted(f for f in glob.glob('/kaggle/working/model_*.pt') if any(f.endswith(f'_s{s}.pt') for s in [3, 4, 5]))
print('preds:', len(g2_p), '(expected 12) | models:', len(g2_m), '(expected 12)')
for f in g2_p:
    z = np.load(f)
    print(os.path.basename(f), 'keys:', len(z.files), '(14) | test_id', z['z_test_id'].shape, '| ext_pad', z['z_ext_pad'].shape, '| ext_ham', z['z_ext_ham'].shape)
out = '/kaggle/working/calib_results_fresh.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in g2_p + g2_m: zf.write(f, os.path.basename(f))
print('zip size MB:', round(os.path.getsize(out) / 1e6, 1))

In [ ]:
# ---- Cell C2: logits of the FRESH-seed models (B0, B3; CE, focal; seeds 3-5) on corrupted calibration images ----
import os, io, numpy as np, pandas as pd, torch, torch.nn as nn, torchvision, torchvision.transforms as T
from PIL import Image, ImageFilter
c2_dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', c2_dev)
def c2_find(name):
    for d in ['/kaggle/working', '/kaggle/input/datasets/cvaravinda/calib-results-fresh', '/kaggle/input/datasets/cvaravinda/calib-results-derived',
              '/kaggle/input/datasets/cvaravinda/calib-results-b3', '/kaggle/input/datasets/cvaravinda/calib-results-all2']:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
c2_df = pd.read_csv(c2_find('final_splits.csv')); c2_cal = c2_df[c2_df.set == 'cal']
c2_base = [T.Resize((224, 224))(Image.open(p).convert('RGB')) for p in c2_cal.path]   # bilinear, as in training
C2_KINDS = ['clean', 'blur', 'jpeg', 'color', 'noise', 'lowres', 'combo']
c2_cj = T.ColorJitter(0.3, 0.3, 0.3, 0.05)

def c2_corrupt(img, kind, rng):
    def blur(im): return im.filter(ImageFilter.GaussianBlur(rng.uniform(1, 2.5)))
    def jpeg(im):
        b = io.BytesIO(); im.save(b, 'JPEG', quality=int(rng.uniform(35, 70))); b.seek(0); return Image.open(b).convert('RGB')
    def color(im): torch.manual_seed(int(rng.randint(1_000_000_000))); return c2_cj(im)
    def noise(im):
        a = np.asarray(im).astype(np.float32) + rng.normal(0, rng.uniform(4, 10), (224, 224, 3))
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    def lowres(im): return im.resize((96, 96), Image.BILINEAR).resize((224, 224), Image.BILINEAR)
    if kind == 'clean': return img
    if kind == 'combo': return lowres(color(jpeg(blur(img))))
    return dict(blur=blur, jpeg=jpeg, color=color, noise=noise, lowres=lowres)[kind](img)

c2_arr = {}
for ki, k in enumerate(C2_KINDS):
    c2_arr[k] = np.stack([np.asarray(c2_corrupt(im, k, np.random.RandomState(10000 * ki + i))) for i, im in enumerate(c2_base)])
print('corrupted arrays:', {k: v.shape for k, v in c2_arr.items()})
c2_mu = torch.tensor([0.485, 0.456, 0.406], device=c2_dev).view(1, 3, 1, 1)
c2_sd = torch.tensor([0.229, 0.224, 0.225], device=c2_dev).view(1, 3, 1, 1)

def c2_model(bb, path):
    if bb == 'B0': m = torchvision.models.efficientnet_b0(weights=None); m.classifier[1] = nn.Linear(1280, 8)
    else:          m = torchvision.models.efficientnet_b3(weights=None); m.classifier[1] = nn.Linear(1536, 8)
    m.load_state_dict(torch.load(path, map_location='cpu')); return m.to(c2_dev).eval()

@torch.no_grad()
def c2_logits(m, arr):
    out = []
    for i in range(0, len(arr), 64):
        x = torch.from_numpy(arr[i:i + 64]).to(c2_dev).permute(0, 3, 1, 2).float().div(255)
        with torch.cuda.amp.autocast(): out.append(m((x - c2_mu) / c2_sd).float().cpu())
    return torch.cat(out).numpy()

c2_res = {}; c2_maxdiff = 0
for bb, pre in [('B0', ''), ('B3', 'b3_')]:
    for v in ['ce', 'focal']:
        for s in [3, 4, 5]:
            m = c2_model(bb, c2_find(f'model_{pre}{v}_s{s}.pt'))
            for k in C2_KINDS: c2_res[f'{bb}_{v}_s{s}__{k}'] = c2_logits(m, c2_arr[k])
            ref = np.load(c2_find(f'preds_{pre}{v}_s{s}.npz'))['z_cal']
            c2_maxdiff = max(c2_maxdiff, float(np.abs(c2_res[f'{bb}_{v}_s{s}__clean'] - ref).max()))
            del m; torch.cuda.empty_cache()
        print('done', bb, v)
np.savez_compressed('/kaggle/working/corr_cal_logits_fresh.npz', **c2_res)
print('keys:', len(c2_res), '(expected 84) | shape:', c2_res['B0_ce_s3__blur'].shape, '(expected (487, 8))')
print('max |clean logits - saved z_cal|:', round(c2_maxdiff, 4), '(should be under about 0.05)')
print('file MB:', round(os.path.getsize('/kaggle/working/corr_cal_logits_fresh.npz') / 1e6, 1))

## 6. Frozen confirmatory analysis of the matcher (paper Section 6.3)
The matcher code and hypotheses H1-H4 with their decision criteria are fixed in this cell and were applied unchanged to seeds 3-5.

In [ ]:
# ---- Cell M3: FROZEN confirmatory analysis on fresh seeds 3-5 (matcher code unchanged from M1) ----
import os, numpy as np, pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import softmax, log_softmax
from scipy.stats import wasserstein_distance, binomtest

def m3_find(name):
    for d in ['/kaggle/working', '/kaggle/input/datasets/cvaravinda/calib-results-fresh', '/kaggle/input/datasets/cvaravinda/calib-results-derived',
              '/kaggle/input/datasets/cvaravinda/calib-results-b3', '/kaggle/input/datasets/cvaravinda/calib-results-all2']:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
def m3_fitT(z, y):
    f = lambda t: -np.mean(log_softmax(z / t, 1)[np.arange(len(y)), y])
    return minimize_scalar(f, bounds=(0.05, 10), method='bounded').x
def m3_ece(conf, acc, nb=15):
    b = (np.ceil(conf * nb) - 1).clip(0, nb - 1).astype(int); e = 0.0
    for k in range(nb):
        m = b == k
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

SETS = ['test_id', 'shift_trunk', 'ext_ham', 'ext_pad']
KINDS = ['clean', 'blur', 'jpeg', 'color', 'noise', 'lowres', 'combo']
CFG = [('B0', '', 'ce'), ('B0', '', 'focal'), ('B3', 'b3_', 'ce'), ('B3', 'b3_', 'focal')]
CZ = np.load(m3_find('corr_cal_logits_fresh.npz')); rows = []
for bb, pre, v in CFG:
    for s in [3, 4, 5]:
        d = np.load(m3_find(f'preds_{pre}{v}_s{s}.npz')); yc = d['y_cal']
        Zc = {k: CZ[f'{bb}_{v}_s{s}__{k}'] for k in KINDS}
        Tk = {k: m3_fitT(Zc[k], yc) for k in KINDS}; mp = {k: softmax(Zc[k], 1).max(1) for k in KINDS}
        for st in SETS:
            z, y = d[f'z_{st}'], d[f'y_{st}']; pr = softmax(z, 1); acc = (pr.argmax(1) == y).astype(float)
            dist = {k: wasserstein_distance(pr.max(1), mp[k]) for k in KINDS}; pick = min(dist, key=dist.get)
            e_clean = m3_ece(softmax(z / Tk['clean'], 1).max(1), acc); e_match = m3_ece(softmax(z / Tk[pick], 1).max(1), acc)
            rows.append(dict(bb=bb, v=v, seed=s, set=st, raw=m3_ece(pr.max(1), acc), cleanTS=e_clean, matched=e_match, pick=pick, delta=e_match - e_clean))
F3 = pd.DataFrame(rows); F3.to_csv('/kaggle/working/m3_rows.csv', index=False)

print('=== fresh seeds 3-5: ECE mean±sd (raw / clean-fit / matched) ===')
for st in SETS:
    for bb, pre, v in CFG:
        d = F3[(F3.set == st) & (F3.bb == bb) & (F3.v == v)]
        print(f'{st:11s} {bb} {v:5s} raw {d.raw.mean():.3f}±{d.raw.std():.3f} | clean {d.cleanTS.mean():.3f}±{d.cleanTS.std():.3f} | matched {d.matched.mean():.3f}±{d.matched.std():.3f}')

print('\n=== per-model detail: PAD-UFES-20 and HAM10000 ===')
print(F3[F3.set.isin(['ext_pad', 'ext_ham'])][['set', 'bb', 'v', 'seed', 'cleanTS', 'matched', 'delta', 'pick']].round(3).to_string(index=False))

print('\n=== FROZEN HYPOTHESES (verdicts follow the criteria fixed before the fresh data existed) ===')
P = F3[F3.set == 'ext_pad']; k1 = int((P.delta < 0).sum()); p1 = binomtest(k1, 12, 0.5, alternative='greater').pvalue
print(f'H1 PAD matched < clean: improved {k1}/12, mean dECE {P.delta.mean():+.3f}, one-sided sign p={p1:.3f} -> ' + ('CONFIRMED' if p1 < 0.05 else 'NOT CONFIRMED'))
H = F3[F3.set == 'ext_ham']; k2 = int((H.delta < 0).sum())
print(f'H2 HAM (report only): improved {k2}/12, mean dECE {H.delta.mean():+.3f}, two-sided sign p={binomtest(k2, 12, 0.5).pvalue:.3f}')
for st in ['ext_ham', 'ext_pad']:
    n = int((F3[F3.set == st].pick != 'clean').sum()); print(f'H3 {st}: non-clean picks {n}/12 -> ' + ('PASS' if n >= 11 else 'FAIL'))
for st in ['test_id', 'shift_trunk']:
    n = int(F3[F3.set == st].pick.isin(['clean', 'color']).sum()); print(f'H3 {st}: clean/color picks {n}/12 -> ' + ('PASS' if n >= 11 else 'FAIL'))
N = P[P.pick == 'noise']
if len(N) < 4: print(f'H4 PAD noise picks: only {len(N)} -> NOT TESTABLE')
else: print(f'H4 PAD noise picks: {int((N.delta < 0).sum())}/{len(N)} improved ({(N.delta < 0).mean():.0%}) -> ' + ('PASS' if (N.delta < 0).mean() >= 0.8 else 'FAIL'))

try:
    old = pd.read_csv(m3_find('m1_rows.csv')); piv = old[old.method.isin(['cleanTS', 'matched'])].pivot_table(index=['bb', 'v', 'seed', 'set'], columns='method', values='ece').reset_index()
    piv['delta'] = piv.matched - piv.cleanTS; allr = pd.concat([piv[['set', 'delta']], F3[['set', 'delta']]])
    print('\n=== pooled over all 24 model-seeds (effect size only, not a test) ===')
    for st in ['ext_pad', 'ext_ham']:
        d = allr[allr.set == st].delta; print(f'{st}: mean dECE {d.mean():+.3f} | improved {int((d < 0).sum())}/24')
except Exception as e: print('pooled summary skipped:', e)

In [ ]:
import os, zipfile
f = ['/kaggle/working/corr_cal_logits_fresh.npz', '/kaggle/working/m3_rows.csv']
with zipfile.ZipFile('/kaggle/working/calib_results_fresh_derived.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    [z.write(x, os.path.basename(x)) for x in f if os.path.exists(x)]
print([(os.path.basename(x), os.path.exists(x)) for x in f], round(os.path.getsize('/kaggle/working/calib_results_fresh_derived.zip') / 1e6, 2), 'MB')

## 7. Grad-CAM deletion/insertion faithfulness, all 24 models, with random and centre orders (paper Sections 3.5 and 6.6)

In [ ]:
# ---- Cell F6: faithfulness (Grad-CAM deletion/insertion) over all 24 models, resumable ----
import os, json, time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, torchvision
import torchvision.transforms as T, torchvision.transforms.functional as TF
from PIL import Image
from scipy.stats import wilcoxon
f6_OUT = '/kaggle/working/faith_all_models.jsonl'
f6_SETS = ['test_id', 'ext_pad', 'ext_ham']; f6_N = 100; f6_STEPS = np.linspace(0, 1, 21)
f6_trap = getattr(np, 'trapezoid', None) or np.trapz
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
def f6_find(name):
    for d in ['/kaggle/working', '/kaggle/input/datasets/cvaravinda/calib-results-fresh', '/kaggle/input/datasets/cvaravinda/calib-results-b3',
              '/kaggle/input/datasets/cvaravinda/calib-results-all2']:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
f6_df = pd.read_csv(f6_find('final_splits.csv'))
f6_MAP = {l: i for i, l in enumerate([0, 1, 3, 4, 7, 8, 9, 10])}
f6_mu = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
f6_sd = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
f6_rs = T.Resize((224, 224))

def f6_model(bb, path):
    if bb == 'B0': m = torchvision.models.efficientnet_b0(weights=None); m.classifier[1] = nn.Linear(1280, 8)
    else:          m = torchvision.models.efficientnet_b3(weights=None); m.classifier[1] = nn.Linear(1536, 8)
    m.load_state_dict(torch.load(path, map_location='cpu')); return m.to(dev).eval()

def f6_load(p):
    im = f6_rs(Image.open(p).convert('RGB'))
    x = torch.from_numpy(np.asarray(im).copy()).permute(2, 0, 1).float().div(255).unsqueeze(0).to(dev)
    return (x - f6_mu) / f6_sd

def f6_cam(m, x):
    with torch.enable_grad():
        a = m.features(x); a.retain_grad(); out = m.classifier(torch.flatten(m.avgpool(a), 1))
        c = int(out.argmax(1)); m.zero_grad(); out[0, c].backward()
        cam = F.relu((a.grad.mean((2, 3), keepdim=True) * a).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)[0, 0]
    return cam.detach(), c

def f6_auc(m, x, base, score, c):
    flat = score.flatten(); rank = torch.empty_like(flat)
    rank[flat.argsort(descending=True)] = torch.arange(flat.numel(), device=dev, dtype=flat.dtype); rank = rank.view(1, 1, 224, 224); res = {}
    with torch.no_grad():
        for tag in ('del', 'ins'):
            xs = []
            for f in f6_STEPS:
                mk = (rank < f * 224 * 224).float(); xs.append(base * mk + x * (1 - mk) if tag == 'del' else x * mk + base * (1 - mk))
            res[tag] = float(f6_trap(F.softmax(m(torch.cat(xs)), 1)[:, c].cpu().numpy(), f6_STEPS))
    return res

f6_samples = {s: f6_df[f6_df.set == s].sample(100, random_state=7).reset_index() for s in f6_SETS}
done = set()
if os.path.exists(f6_OUT):
    for l in open(f6_OUT): j = json.loads(l); done.add((j['tag'], j['set'], j['i']))
t0 = time.time(); stop = False; n = 0
for bb, pre in [('B0', ''), ('B3', 'b3_')]:
    for v in ['ce', 'focal']:
        for sd in range(6):
            tag = f'{bb}_{v}_s{sd}'
            if all((tag, s, i) in done for s in f6_SETS for i in range(f6_N)): continue
            if time.time() - t0 > 170 * 60: stop = True; break
            m = f6_model(bb, f6_find(f'model_{pre}{v}_s{sd}.pt'))
            for s in f6_SETS:
                for i, r in f6_samples[s].iterrows():
                    if (tag, s, i) in done: continue
                    x = f6_load(r['path']); cam, c = f6_cam(m, x)
                    rnd = torch.rand(224, 224, device=dev, generator=torch.Generator(device=dev).manual_seed(1000 + i))
                    rec = dict(tag=tag, set=s, i=int(i), correct=int(c == f6_MAP.get(int(r['label']), -1)))
                    for ft, base in (('g', torch.zeros_like(x)), ('b', TF.gaussian_blur(x, kernel_size=[51, 51], sigma=[15.0, 15.0]))):
                        a = f6_auc(m, x, base, cam, c); b = f6_auc(m, x, base, rnd, c)
                        rec.update({f'cam_del_{ft}': a['del'], f'cam_ins_{ft}': a['ins'], f'rnd_del_{ft}': b['del'], f'rnd_ins_{ft}': b['ins']})
                    open(f6_OUT, 'a').write(json.dumps(rec) + '\n'); n += 1
            print('done', tag, round(time.time() - t0), 's'); del m; torch.cuda.empty_cache()
        if stop: break
    if stop: break
if stop: print('time limit reached: run this cell again to resume')

R = pd.DataFrame([json.loads(l) for l in open(f6_OUT)]); print('\nlines:', len(R), '| models:', R.tag.nunique())
print('\n=== in how many models does Grad-CAM beat random order? ===')
for st in f6_SETS:
    for ft, fn in (('g', 'grey'), ('b', 'blur')):
        for dr in ('del', 'ins'):
            ks = sig = 0; diffs = []
            for tag, g in R[R.set == st].groupby('tag'):
                if len(g) < 20: continue
                d = g[f'cam_{dr}_{ft}'] - g[f'rnd_{dr}_{ft}']; diffs.append(d.mean())
                better = d.mean() < 0 if dr == 'del' else d.mean() > 0; ks += better
                sig += bool(better and wilcoxon(d).pvalue < 0.05)
            print(f'{st:8s} {fn:5s} {dr}: better in {ks}/{len(diffs)} models | significant (p<.05) in {sig} | mean diff {np.mean(diffs):+.3f}, median {np.median(diffs):+.3f}')

In [ ]:
import zipfile, os
with zipfile.ZipFile('/kaggle/working/faith_all_models.zip', 'w', zipfile.ZIP_DEFLATED) as z: z.write('/kaggle/working/faith_all_models.jsonl', 'faith_all_models.jsonl')
print(round(os.path.getsize('/kaggle/working/faith_all_models.zip') / 1e6, 2), 'MB')

In [ ]:
# ---- Cell C3: centre-bias control for Grad-CAM faithfulness (same 100 images/set and 24 models as F6), resumable, GPU ----
import os, json, time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, torchvision
import torchvision.transforms as T, torchvision.transforms.functional as TF
from PIL import Image
from scipy.stats import wilcoxon
c3_OUT = '/kaggle/working/faith_centre.jsonl'; c3_SETS = ['test_id', 'ext_pad', 'ext_ham']; c3_N = 100; c3_STEPS = np.linspace(0, 1, 21)
c3_trap = getattr(np, 'trapezoid', None) or np.trapz
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
c3_DIRS = ['/kaggle/working', '/kaggle/input/datasets/cvaravinda/faith-all-models', '/kaggle/input/datasets/cvaravinda/calib-results-fresh',
           '/kaggle/input/datasets/cvaravinda/calib-results-b3', '/kaggle/input/datasets/cvaravinda/calib-results-all2']
def c3_find(name):
    for d in c3_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
c3_df = pd.read_csv(c3_find('final_splits.csv'))
c3_mu = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); c3_sd = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
c3_rs = T.Resize((224, 224))
c3_yy, c3_xx = torch.meshgrid(torch.arange(224, device=dev).float(), torch.arange(224, device=dev).float(), indexing='ij')
c3_score = -((c3_yy - 111.5) ** 2 + (c3_xx - 111.5) ** 2)          # highest score at the image centre -> centre pixels are removed (or revealed) first
def c3_model(bb, path):
    if bb == 'B0': m = torchvision.models.efficientnet_b0(weights=None); m.classifier[1] = nn.Linear(1280, 8)
    else:          m = torchvision.models.efficientnet_b3(weights=None); m.classifier[1] = nn.Linear(1536, 8)
    m.load_state_dict(torch.load(path, map_location='cpu')); return m.to(dev).eval()
def c3_load(p):
    a = np.asarray(c3_rs(Image.open(p).convert('RGB'))).copy()
    return (torch.from_numpy(a).permute(2, 0, 1).float().div(255).unsqueeze(0).to(dev) - c3_mu) / c3_sd
def c3_auc(m, x, base, c):
    flat = c3_score.flatten(); rank = torch.empty_like(flat); rank[flat.argsort(descending=True)] = torch.arange(flat.numel(), device=dev, dtype=flat.dtype); rank = rank.view(1, 1, 224, 224); res = {}
    with torch.no_grad():
        for tag in ('del', 'ins'):
            xs = []
            for f in c3_STEPS:
                mk = (rank < f * 224 * 224).float(); xs.append(base * mk + x * (1 - mk) if tag == 'del' else x * mk + base * (1 - mk))
            res[tag] = float(c3_trap(F.softmax(m(torch.cat(xs)), 1)[:, c].cpu().numpy(), c3_STEPS))
    return res
c3_samples = {s: c3_df[c3_df.set == s].sample(100, random_state=7).reset_index() for s in c3_SETS}
done = set()
if os.path.exists(c3_OUT):
    for l in open(c3_OUT): j = json.loads(l); done.add((j['tag'], j['set'], j['i']))
t0 = time.time(); stop = False
for bb, pre in [('B0', ''), ('B3', 'b3_')]:
    for v in ['ce', 'focal']:
        for sd in range(6):
            tag = f'{bb}_{v}_s{sd}'
            if all((tag, s, i) in done for s in c3_SETS for i in range(c3_N)): continue
            if time.time() - t0 > 170 * 60: stop = True; break
            m = c3_model(bb, c3_find(f'model_{pre}{v}_s{sd}.pt'))
            for s in c3_SETS:
                for i, r in c3_samples[s].iterrows():
                    if (tag, s, i) in done: continue
                    x = c3_load(r['path'])
                    with torch.no_grad(): c = int(m(x).argmax(1))
                    rec = dict(tag=tag, set=s, i=int(i))
                    for ft, base in (('g', torch.zeros_like(x)), ('b', TF.gaussian_blur(x, kernel_size=[51, 51], sigma=[15.0, 15.0]))):
                        a = c3_auc(m, x, base, c); rec.update({f'cen_del_{ft}': a['del'], f'cen_ins_{ft}': a['ins']})
                    open(c3_OUT, 'a').write(json.dumps(rec) + '\n')
            print('done', tag, round(time.time() - t0), 's'); del m; torch.cuda.empty_cache()
        if stop: break
    if stop: break
if stop: print('time limit reached: run this cell again to resume')

C = pd.DataFrame([json.loads(l) for l in open(c3_OUT)]); Fa = pd.DataFrame([json.loads(l) for l in open(c3_find('faith_all_models.jsonl'))])
M = Fa.merge(C, on=['tag', 'set', 'i']); print('\nmerged rows:', len(M), '| models:', M.tag.nunique())
print('\n=== number of models in which Grad-CAM beats the CENTRE order (significant at p<.05 in brackets) | and centre beats RANDOM ===')
for st in c3_SETS:
    for ft, fn in (('g', 'grey'), ('b', 'blur')):
        for dr in ('del', 'ins'):
            k1 = s1 = k2 = s2 = n = 0
            for tag, g in M[M.set == st].groupby('tag'):
                if len(g) < 20: continue
                n += 1; d1 = g[f'cam_{dr}_{ft}'] - g[f'cen_{dr}_{ft}']; d2 = g[f'cen_{dr}_{ft}'] - g[f'rnd_{dr}_{ft}']
                b1 = d1.mean() < 0 if dr == 'del' else d1.mean() > 0; b2 = d2.mean() < 0 if dr == 'del' else d2.mean() > 0
                k1 += int(b1); s1 += int(b1 and wilcoxon(d1).pvalue < 0.05); k2 += int(b2); s2 += int(b2 and wilcoxon(d2).pvalue < 0.05)
            print(f'{st:8s} {fn:5s} {dr}: Grad-CAM better than centre in {k1}/{n} ({s1}) | centre better than random in {k2}/{n} ({s2}) | mean AUC cam {M[M.set==st][f"cam_{dr}_{ft}"].mean():.3f} centre {M[M.set==st][f"cen_{dr}_{ft}"].mean():.3f} random {M[M.set==st][f"rnd_{dr}_{ft}"].mean():.3f}')

## 8. Final tables (paper Tables 2-7, Supplementary Tables S1-S2)

In [ ]:
# ---- Cell T1 (corrected): final LaTeX tables + CSVs from the saved files (seeds 0-5), CPU ----
import os, json, zipfile, numpy as np, pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import softmax, log_softmax
from scipy.stats import binomtest, wilcoxon

T1_DIRS = ['/kaggle/working', '/kaggle/input/datasets/cvaravinda/calib-results-derived', '/kaggle/input/datasets/cvaravinda/lib-results-fresh-derived',
           '/kaggle/input/datasets/cvaravinda/faith-all-models', '/kaggle/input/datasets/cvaravinda/calib-results-fresh',
           '/kaggle/input/datasets/cvaravinda/calib-results-b3', '/kaggle/input/datasets/cvaravinda/calib-results-all2']
def t1_find(name):
    for d in T1_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
def t1_fitT(z, y):
    f = lambda t: -np.mean(log_softmax(z / t, 1)[np.arange(len(y)), y]); return minimize_scalar(f, bounds=(0.05, 10), method='bounded').x
def t1_ece(p, y, nb=15):
    conf, acc = p.max(1), (p.argmax(1) == y).astype(float); e = 0.0; b = np.linspace(0, 1, nb + 1)
    for lo, hi in zip(b[:-1], b[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e
def t1_nll(p, y): return float(-np.mean(np.log(p[np.arange(len(y)), y] + 1e-12)))

SETS = ['test_id', 'shift_trunk', 'ext_ham', 'ext_pad']; EXT = ('ext_ham', 'ext_pad')
SN = {'test_id': 'In-domain', 'shift_trunk': 'Trunk site', 'ext_ham': 'HAM10000', 'ext_pad': 'PAD-UFES-20'}
CFG = [('B0', '', 'ce'), ('B0', '', 'focal'), ('B3', 'b3_', 'ce'), ('B3', 'b3_', 'focal')]
CN = {('B0', 'ce'): 'EfficientNet-B0, CE', ('B0', 'focal'): 'EfficientNet-B0, focal', ('B3', 'ce'): 'EfficientNet-B3, CE', ('B3', 'focal'): 'EfficientNet-B3, focal'}
rng = np.random.RandomState(0); rows, ens = [], []
for bb, pre, v in CFG:
    D = {s: np.load(t1_find(f'preds_{pre}{v}_s{s}.npz')) for s in range(6)}
    Ts = {s: t1_fitT(D[s]['z_cal'], D[s]['y_cal']) for s in range(6)}
    pc = np.mean([softmax(D[s]['z_cal'], 1) for s in range(6)], 0); Te = t1_fitT(np.log(pc + 1e-12), D[0]['y_cal'])
    for st in SETS:
        for s in range(6):
            z, y = D[s][f'z_{st}'], D[s][f'y_{st}']
            for m, T in (('raw', 1.0), ('cleanTS', Ts[s])):
                p = softmax(z / T, 1); rows.append(dict(bb=bb, v=v, seed=s, set=st, method=m, acc=float((p.argmax(1) == y).mean()), ece=t1_ece(p, y), nll=t1_nll(p, y)))
            if st in EXT:
                o = []
                for r in range(20):
                    idx = rng.permutation(len(y)); a, b = idx[:100], idx[100:]; p = softmax(z[b] / t1_fitT(z[a], y[a]), 1); o.append((t1_ece(p, y[b]), t1_nll(p, y[b])))
                rows.append(dict(bb=bb, v=v, seed=s, set=st, method='oracle100', acc=np.nan, ece=np.mean([x[0] for x in o]), nll=np.mean([x[1] for x in o])))
        y = D[0][f'y_{st}']; pe = np.mean([softmax(D[s][f'z_{st}'], 1) for s in range(6)], 0); pt = softmax(np.log(pe + 1e-12) / Te, 1)
        ens.append(dict(bb=bb, v=v, set=st, method='ens_TS', acc=float((pt.argmax(1) == y).mean()), ece=t1_ece(pt, y), nll=t1_nll(pt, y)))
R = pd.DataFrame(rows); E = pd.DataFrame(ens); os.makedirs('/kaggle/working/tables', exist_ok=True)
R.to_csv('/kaggle/working/tables/ladder_rows.csv', index=False); E.to_csv('/kaggle/working/tables/ladder_ens.csv', index=False)

def pm(a): return f'{np.mean(a):.3f}\\,$\\pm$\\,{np.std(a, ddof=1):.3f}'
def t1_ladder(metric, cap, label, fname):
    L = ['\\begin{table*}[t]', '\\centering', '\\small', f'\\caption{{{cap}}}', f'\\label{{{label}}}', '\\begin{tabular}{llcccc}', '\\toprule',
         'Model & Method & ' + ' & '.join(SN[s] for s in SETS) + ' \\\\', '\\midrule']
    for ci, (bb, pre, v) in enumerate(CFG):
        for mi, (m, ml) in enumerate([('raw', 'Raw'), ('cleanTS', 'Clean-fit TS'), ('ens_TS', 'Six-seed ensemble + TS'), ('oracle100', 'Target-fit TS (100 labels)')]):
            cells = []
            for st in SETS:
                if m == 'ens_TS': cells.append(f"{E[(E.bb == bb) & (E.v == v) & (E.set == st)][metric].iloc[0]:.3f}")
                elif m == 'oracle100' and st not in EXT: cells.append('--')
                else: cells.append(pm(R[(R.bb == bb) & (R.v == v) & (R.set == st) & (R.method == m)][metric].values))
            L.append((CN[(bb, v)] if mi == 0 else '') + ' & ' + ml + ' & ' + ' & '.join(cells) + ' \\\\')
        if ci < 3: L.append('\\midrule')
    L += ['\\bottomrule', '\\end{tabular}', '\\end{table*}']; open(f'/kaggle/working/tables/{fname}', 'w').write('\n'.join(L)); return '\n'.join(L)

def t1_acc():
    L = ['\\begin{table}[t]', '\\centering', '\\small', '\\caption{Top-1 accuracy, mean $\\pm$ s.d. over six seeds.}', '\\label{tab:acc}', '\\begin{tabular}{lcccc}', '\\toprule',
         'Model & ' + ' & '.join(SN[s] for s in SETS) + ' \\\\', '\\midrule']
    for bb, pre, v in CFG:
        L.append(CN[(bb, v)] + ' & ' + ' & '.join(pm(R[(R.bb == bb) & (R.v == v) & (R.set == st) & (R.method == 'raw')].acc.values) for st in SETS) + ' \\\\')
    L += ['\\bottomrule', '\\end{tabular}', '\\end{table}']; open('/kaggle/working/tables/tab_acc.tex', 'w').write('\n'.join(L)); return '\n'.join(L)

def t1_matcher():
    m1 = pd.read_csv(t1_find('m1_rows.csv')); pv = m1[m1.method.isin(['cleanTS', 'matched'])].pivot_table(index=['bb', 'v', 'seed', 'set'], columns='method', values='ece').reset_index()
    pv['delta'] = pv.matched - pv.cleanTS; pv = pv.merge(m1[m1.method == 'matched'][['bb', 'v', 'seed', 'set', 'pick']], on=['bb', 'v', 'seed', 'set'])
    m3 = pd.read_csv(t1_find('m3_rows.csv'))
    cohorts = [('Exploratory (seeds 0--2)', pv), ('Confirmatory (seeds 3--5)', m3)]
    L = ['\\begin{table}[t]', '\\centering', '\\small', '\\caption{Label-free matched scaling versus clean-fit scaling. $\\Delta$ECE is matched minus clean-fit (negative favours matching); Improved counts models whose ECE improved, out of those whose ECE changed; ties (the matcher picked the clean fit) are excluded, and $p$ is a two-sided sign test on the remaining models (-- if fewer than six). Alarm counts the models whose matcher pick is non-clean on the external sets, and clean or colour on the in-distribution sets.}',
         '\\label{tab:matcher}', '\\begin{tabular}{llcccc}', '\\toprule', 'Cohort & Set & Improved & Mean $\\Delta$ECE & $p$ & Alarm \\\\', '\\midrule']
    for ci, (cn, d) in enumerate(cohorts):
        for st in SETS:
            g = d[d.set == st]; nz = g[g.delta.abs() > 1e-9]; k = int((nz.delta < 0).sum()); n = len(nz)
            al = int((g.pick != 'clean').sum()) if st in EXT else int(g.pick.isin(['clean', 'color']).sum())
            ptxt = f'{binomtest(k, n, 0.5).pvalue:.3f}' if n >= 6 else '--'
            L.append((cn if st == SETS[0] else '') + f" & {SN[st]} & {k}/{n} & ${g.delta.mean():+.3f}$ & {ptxt} & {al}/{len(g)} \\\\")
        if ci == 0: L.append('\\midrule')
    L += ['\\bottomrule', '\\end{tabular}', '\\end{table}']; open('/kaggle/working/tables/tab_matcher.tex', 'w').write('\n'.join(L)); return '\n'.join(L)

def t1_faith():
    F = pd.DataFrame([json.loads(l) for l in open(t1_find('faith_all_models.jsonl'))]); nm = F.tag.nunique()
    L = ['\\begin{table}[t]', '\\centering', '\\small', f'\\caption{{Grad-CAM faithfulness against random pixel order across {nm} models (100 images per set per model). Cells give the number of models in which Grad-CAM is better (and, in brackets, significantly better, Wilcoxon $p<0.05$ per model).}}',
         '\\label{tab:faith}', '\\begin{tabular}{lccc}', '\\toprule', 'Test & ' + ' & '.join(SN[s] for s in ['test_id', 'ext_pad', 'ext_ham']) + ' \\\\', '\\midrule']
    for fn, ft, dr, lab in [('grey', 'g', 'ins', 'Insertion, grey fill'), ('blur', 'b', 'del', 'Deletion, blurred fill'), ('blur', 'b', 'ins', 'Insertion, blurred fill'), ('grey', 'g', 'del', 'Deletion, grey fill')]:
        cells = []
        for st in ['test_id', 'ext_pad', 'ext_ham']:
            ks = sg = 0; n = 0
            for tag, g in F[F.set == st].groupby('tag'):
                if len(g) < 20: continue
                d = g[f'cam_{dr}_{ft}'] - g[f'rnd_{dr}_{ft}']; better = d.mean() < 0 if dr == 'del' else d.mean() > 0; ks += int(better); n += 1; sg += int(better and wilcoxon(d).pvalue < 0.05)
            cells.append(f'{ks}/{n} ({sg})')
        L.append(lab + ' & ' + ' & '.join(cells) + ' \\\\')
    L += ['\\bottomrule', '\\end{tabular}', '\\end{table}']; open('/kaggle/working/tables/tab_faith.tex', 'w').write('\n'.join(L)); return '\n'.join(L)

out = [t1_ladder('ece', 'ECE (15 bins), mean $\\pm$ s.d. over six seeds. The ensemble has no s.d.; the target-fit rows (temperature fitted on 100 labelled target images) exist only for the external sets.', 'tab:ece', 'tab_ece.tex'),
       t1_ladder('nll', 'Negative log-likelihood, mean $\\pm$ s.d. over six seeds.', 'tab:nll', 'tab_nll.tex'), t1_acc(), t1_matcher(), t1_faith()]
with zipfile.ZipFile('/kaggle/working/tables.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir('/kaggle/working/tables')): z.write(f'/kaggle/working/tables/{f}', f)
print('files:', sorted(os.listdir('/kaggle/working/tables')), '\n')
print('\n\n'.join(out))

## 9. ECE binning sensitivity (Supplementary Table S3)

In [ ]:
# ---- Cell E2: ECE binning sensitivity (10/15/20 equal-width, 15 equal-mass), seeds 0-5, CPU ----
import os, numpy as np, pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import softmax, log_softmax
E2_DIRS = ['/kaggle/working', '/kaggle/input/datasets/cvaravinda/calib-results-fresh', '/kaggle/input/datasets/cvaravinda/calib-results-b3', '/kaggle/input/datasets/cvaravinda/calib-results-all2']
def e2_find(name):
    for d in E2_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
def e2_fitT(z, y):
    f = lambda t: -np.mean(log_softmax(z / t, 1)[np.arange(len(y)), y]); return minimize_scalar(f, bounds=(0.05, 10), method='bounded').x
def e2_ew(conf, acc, nb):
    b = (np.ceil(conf * nb) - 1).clip(0, nb - 1).astype(int); e = 0.0
    for k in range(nb):
        m = b == k
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e
def e2_em(conf, acc, nb=15):
    o = np.argsort(conf); e = 0.0
    for idx in np.array_split(o, nb):
        if len(idx): e += len(idx) / len(conf) * abs(acc[idx].mean() - conf[idx].mean())
    return e
BINS = [('ew10', lambda c, a: e2_ew(c, a, 10)), ('ew15', lambda c, a: e2_ew(c, a, 15)), ('ew20', lambda c, a: e2_ew(c, a, 20)), ('em15', lambda c, a: e2_em(c, a, 15))]
def e2_all(p, y):
    conf, acc = p.max(1), (p.argmax(1) == y).astype(float); return {k: f(conf, acc) for k, f in BINS}
SETS = ['test_id', 'shift_trunk', 'ext_ham', 'ext_pad']; EXT = ('ext_ham', 'ext_pad')
CFG = [('B0', '', 'ce'), ('B0', '', 'focal'), ('B3', 'b3_', 'ce'), ('B3', 'b3_', 'focal')]
rng = np.random.RandomState(0); rows = []
for bb, pre, v in CFG:
    D = {s: np.load(e2_find(f'preds_{pre}{v}_s{s}.npz')) for s in range(6)}; Ts = {s: e2_fitT(D[s]['z_cal'], D[s]['y_cal']) for s in range(6)}
    pc = np.mean([softmax(D[s]['z_cal'], 1) for s in range(6)], 0); Te = e2_fitT(np.log(pc + 1e-12), D[0]['y_cal'])
    for st in SETS:
        for s in range(6):
            z, y = D[s][f'z_{st}'], D[s][f'y_{st}']
            for m, T in (('raw', 1.0), ('cleanTS', Ts[s])): rows.append(dict(cfg=f'{bb}-{v}', set=st, seed=s, method=m, **e2_all(softmax(z / T, 1), y)))
            if st in EXT:
                acc = {k: [] for k, _ in BINS}
                for r in range(20):
                    idx = rng.permutation(len(y)); a, b = idx[:100], idx[100:]; o = e2_all(softmax(z[b] / e2_fitT(z[a], y[a]), 1), y[b])
                    for k in acc: acc[k].append(o[k])
                rows.append(dict(cfg=f'{bb}-{v}', set=st, seed=s, method='target100', **{k: np.mean(x) for k, x in acc.items()}))
        y = D[0][f'y_{st}']; pe = np.mean([softmax(D[s][f'z_{st}'], 1) for s in range(6)], 0); rows.append(dict(cfg=f'{bb}-{v}', set=st, seed=-1, method='ens', **e2_all(softmax(np.log(pe + 1e-12) / Te, 1), y)))
R = pd.DataFrame(rows); os.makedirs('/kaggle/working/tables', exist_ok=True); R.to_csv('/kaggle/working/tables/ece_binning.csv', index=False)
G = R.groupby(['cfg', 'set', 'method'])[[k for k, _ in BINS]].mean().reset_index()
pd.set_option('display.width', 200); print('=== mean ECE (over six seeds) under four binning schemes ===')
for cfg in G.cfg.unique():
    for st in SETS:
        g = G[(G.cfg == cfg) & (G.set == st)].set_index('method'); order = [m for m in ['raw', 'cleanTS', 'ens', 'target100'] if m in g.index]
        print(f'\n{cfg} | {st}'); print(g.loc[order, [k for k, _ in BINS]].round(3).to_string())
print('\n=== does the ranking of methods depend on the binning? (ranking by mean ECE, smallest first) ===')
chg = 0; tot = 0
for (cfg, st), g in G.groupby(['cfg', 'set']):
    g = g.set_index('method'); rk = {k: tuple(g[k].sort_values().index) for k, _ in BINS}; tot += 1
    if len(set(rk.values())) > 1: chg += 1; print(f'{cfg} {st}:', {k: ' < '.join(v) for k, v in rk.items()})
print(f'cases where the ranking is identical under all four binnings: {tot - chg}/{tot}')
print('\n=== clean-fit TS lowers ECE vs raw, per seed (count of 6 seeds) ===')
P = R[R.method.isin(['raw', 'cleanTS'])].pivot_table(index=['cfg', 'set', 'seed'], columns='method', values=[k for k, _ in BINS])
for k, _ in BINS: print(k, ' | '.join(f'{c}/{s}: {int((P[(k, "cleanTS")].loc[(c, s)] < P[(k, "raw")].loc[(c, s)]).sum())}/6' for c in G.cfg.unique() for s in ('test_id', 'ext_pad')))

## 10. Data provenance checks (paper Section 5.1)

In [ ]:
# ---- Cell P1: label provenance + class counts per set + why PAD-UFES-20 accuracy is low (CPU, reads saved files only) ----
import os, numpy as np, pandas as pd
P1_DIRS = ['/kaggle/working', '/kaggle/input/datasets/cvaravinda/calib-results-all2', '/kaggle/input/datasets/cvaravinda/calib-results-fresh']
def p1_find(name):
    for d in P1_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
KEEP = [0, 1, 3, 4, 7, 8, 9, 10]; MAP = {l: i for i, l in enumerate(KEEP)}
df = pd.read_csv(p1_find('final_splits.csv')); print('columns:', list(df.columns)); print('rows per set:\n', df.set.value_counts().to_string(), '\n')
txt = [c for c in df.columns if c not in ('path', 'label', 'set', 'group', 'y') and not pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() < 40]
print('text columns that may hold diagnosis names:', txt)
for c in txt[:3]:
    print(f'\n--- cross-tab of label (raw id) vs {c} (all sets) ---'); print(pd.crosstab(df['label'], df[c]).to_string())
print('\n=== raw label id counts per set (ids not in KEEP are marked *) ===')
ct = pd.crosstab(df['label'], df['set']); ct.index = [f'{i}{"" if i in MAP else " *"}' for i in ct.index]; print(ct.to_string())
print('\n=== rows whose label is NOT one of the 8 kept classes, per set ===')
for s in df.set.unique():
    g = df[df.set == s]; n = (~g['label'].isin(KEEP)).sum(); print(f'{s:12s} {n:5d} of {len(g):5d}')
d = np.load(p1_find('preds_ce_s0.npz'))
print('\n=== prediction files: number of rows used per set (model evaluation) ===')
for s in ['test_id', 'shift_trunk', 'near_ood', 'ext_ham', 'ext_pad']:
    if f'y_{s}' in d: print(f'{s:12s} y={len(d[f"y_{s}"]):5d} | csv rows={int((df.set == s).sum()):5d} | classes present: {sorted(set(d[f"y_{s}"].tolist()))}')
print('\n=== B0 CE seed 0: per-class accuracy and share of predictions, PAD-UFES-20 and HAM10000 ===')
for s in ['ext_pad', 'ext_ham']:
    z, y = d[f'z_{s}'], d[f'y_{s}']; pr = z.argmax(1); n = len(y); tab = []
    for k in range(8):
        m = y == k; tab.append(dict(cls=k, support=int(m.sum()), share=round(m.mean(), 3), recall=round(float((pr[m] == k).mean()), 3) if m.any() else np.nan, predicted_share=round(float((pr == k).mean()), 3)))
    print(f'\n{s}: accuracy {np.mean(pr == y):.3f}'); print(pd.DataFrame(tab).to_string(index=False))
    pres = [k for k in range(8) if (y == k).any()]; print('classes present:', pres, '| macro recall over present classes:', round(float(np.nanmean([t['recall'] for t in tab])), 3))

In [ ]:
# ---- Cell P2: image modality / source check per set (CPU) ----
import os, numpy as np, pandas as pd
from PIL import Image
P2_DIRS = ['/kaggle/working', '/kaggle/input/datasets/cvaravinda/calib-results-all2']
def p2_find(name):
    for d in P2_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
df = pd.read_csv(p2_find('final_splits.csv'))
for s in ['train', 'cal', 'test_id', 'shift_trunk', 'near_ood', 'ext_ham', 'ext_pad']:
    g = df[df.set == s]; print(f'\n=== {s}: {len(g)} images ===')
    print('parent folders (top 5):', g.path.map(lambda p: os.path.dirname(p)).value_counts().head(5).to_dict())
    print('sample paths:', g.path.sample(3, random_state=1).tolist())
    sz = [Image.open(p).size for p in g.path.sample(min(40, len(g)), random_state=1)]; w = np.array([a for a, b in sz]); h = np.array([b for a, b in sz])
    print(f'image size (w x h), 40 random: median {int(np.median(w))} x {int(np.median(h))} | min {w.min()} x {h.min()} | max {w.max()} x {h.max()} | distinct sizes {len(set(sz))}')
print('\nlesion_id examples:', df.lesion_id.head(3).tolist(), '| distinct lesions in train:', df[df.set == 'train'].lesion_id.nunique())

In [ ]:
# ---- Cell P3: which MILK10k image was used per lesion, image-type metadata, ID overlap with HAM10000 (CPU) ----
import os, glob, numpy as np, pandas as pd
P3_DIRS = ['/kaggle/working', '/kaggle/input/datasets/cvaravinda/calib-results-all2']
def p3_find(name):
    for d in P3_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
ROOT = '/kaggle/input/datasets/cvaravinda/skinmilk'
df = pd.read_csv(p3_find('final_splits.csv')); mk = df[df.set.isin(['train', 'cal', 'test_id', 'shift_trunk', 'near_ood'])].copy()
mk['dir'] = mk.path.map(os.path.dirname); mk['stem'] = mk.path.map(lambda p: os.path.splitext(os.path.basename(p))[0])
print('MILK10k rows:', len(mk), '| distinct lesion folders:', mk.dir.nunique())
nf = mk.dir.drop_duplicates().map(lambda d: len([f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]))
print('image files per lesion folder (all lesions):', nf.value_counts().to_dict())
for d in mk.dir.drop_duplicates().sample(4, random_state=3):
    print(' ', os.path.basename(d), '->', [(f, os.path.getsize(os.path.join(d, f)) // 1000) for f in sorted(os.listdir(d))], '| chosen:', os.path.basename(mk[mk.dir == d].path.iloc[0]))
csvs = glob.glob(f'{ROOT}/**/*.csv', recursive=True); print('\nmetadata CSV files under skinmilk:', [os.path.relpath(c, ROOT) for c in csvs][:12])
for c in csvs[:8]:
    try:
        m = pd.read_csv(c, nrows=3); print(' ', os.path.relpath(c, ROOT), '| columns:', list(m.columns)[:20])
    except Exception as e: print(' ', c, 'unreadable', e)
done = False
for c in csvs:
    m = pd.read_csv(c)
    ty = [k for k in m.columns if 'type' in k.lower() or 'modal' in k.lower()]; idc = [k for k in m.columns if 'isic' in k.lower() and 'id' in k.lower()]
    if ty and idc:
        t = m[[idc[0], ty[0]]].drop_duplicates(); j = mk.merge(t, left_on='stem', right_on=idc[0], how='left')
        print(f'\n=== image type of the chosen file ({os.path.relpath(c, ROOT)}, column {ty[0]}) per set ==='); print(pd.crosstab(j.set, j[ty[0]].fillna('NOT FOUND')).to_string())
        print('all image types in the metadata:', m[ty[0]].value_counts().to_dict()); done = True; break
if not done: print('\nno metadata CSV with an image-type column found; send the CSV column list above')
ham = set(df[df.set == 'ext_ham'].path.map(lambda p: os.path.splitext(os.path.basename(p))[0])); ov = ham & set(mk.stem)
print(f'\nexact ISIC-ID overlap between MILK10k files used and HAM10000: {len(ov)} of {len(mk)} MILK rows | examples {sorted(ov)[:5]}')
print('MILK10k ID range:', mk.stem.min(), mk.stem.max(), '| HAM10000 ID range:', min(ham), max(ham))

In [ ]:
# ---- Cell P3b: image type of the chosen file per set + licence/attribution counts, from MILK10k Training metadata (CPU) ----
import os, numpy as np, pandas as pd
P3_DIRS = ['/kaggle/working', '/kaggle/input/datasets/cvaravinda/calib-results-all2']
def p3_find(name):
    for d in P3_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
META = '/kaggle/input/datasets/cvaravinda/skinmilk/MILK10_2024/MILK10k_Training_Metadata.csv'
df = pd.read_csv(p3_find('final_splits.csv')); mk = df[df.set.isin(['train', 'cal', 'test_id', 'shift_trunk', 'near_ood'])].copy()
mk['stem'] = mk.path.map(lambda p: os.path.splitext(os.path.basename(p))[0])
m = pd.read_csv(META); print('metadata rows:', len(m), '| lesions:', m.lesion_id.nunique(), '| columns:', list(m.columns)[:10])
j = mk.merge(m, left_on='stem', right_on='isic_id', how='left', suffixes=('', '_m'))
print('\n=== image type of the file used (per set) ===')
print(pd.crosstab(j.set, j.image_type.fillna('NOT FOUND')).to_string())
print('\n=== licence of the files used (per set) ===');  print(pd.crosstab(j.set, j.copyright_license.fillna('NOT FOUND')).to_string())
print('\n=== attribution (institution) of the files used, all MILK rows ==='); print(j.attribution.fillna('NOT FOUND').value_counts().head(12).to_string())
print('\n=== image_manipulation flags ==='); print(j.image_manipulation.fillna('none/NA').value_counts().head(8).to_string())
print('\n=== site (anatomical) per set: top 8 ==='); print(pd.crosstab(j.site.fillna('NA'), j.set).sort_values('train', ascending=False).head(8).to_string())
print('\n=== skin tone class ==='); print(j.skin_tone_class.fillna('NA').value_counts().to_string())
print('\n=== age_approx and sex (all MILK rows) ==='); print(j.age_approx.describe().round(1).to_string()); print(j.sex.fillna('NA').value_counts().to_string())

## 11. Figures (paper Figs. 2-7 and Supplementary Fig. S1)

In [ ]:
# ---- Cell V3: final figures 1-5 (fixed titles), written to a NEW folder /kaggle/working/figs_final, CPU ----
import os, glob, json, zipfile, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from scipy.special import softmax, log_softmax
V2_DIRS = ['/kaggle/working/tables', '/kaggle/working', '/kaggle/input/datasets/cvaravinda/calib-results-derived', '/kaggle/input/datasets/cvaravinda/lib-results-fresh-derived',
           '/kaggle/input/datasets/cvaravinda/faith-all-models', '/kaggle/input/datasets/cvaravinda/faith-centre-jsonl-new', '/kaggle/input/datasets/cvaravinda/calib-results-all2',
           '/kaggle/input/datasets/cvaravinda/calib-results-b3', '/kaggle/input/datasets/cvaravinda/calib-results-fresh']
def v2_find(pattern):
    for d in V2_DIRS:
        h = sorted(glob.glob(os.path.join(d, pattern)))
        if h: return h[0]
    raise FileNotFoundError(pattern + '  (if this is ladder_rows*.csv, rerun cell T1 first)')
V2_OUT = '/kaggle/working/figs_final'; os.makedirs(V2_OUT, exist_ok=True)
plt.rcParams.update({'font.family': 'serif', 'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.6, 'savefig.dpi': 300, 'pdf.fonttype': 42, 'legend.fontsize': 7})
OI = dict(blue='#0072B2', orange='#E69F00', green='#009E73', verm='#D55E00', sky='#56B4E9', purple='#CC79A7', grey='#8c8c8c')
SETS = ['test_id', 'shift_trunk', 'ext_ham', 'ext_pad']; SN = {'test_id': 'In-domain', 'shift_trunk': 'Trunk site', 'ext_ham': 'HAM10000', 'ext_pad': 'PAD-UFES-20'}
def v2_save(fig, name):
    fig.savefig(f'{V2_OUT}/{name}.pdf', bbox_inches='tight'); fig.savefig(f'{V2_OUT}/{name}.png', bbox_inches='tight'); plt.close(fig); print('saved', name)

# ---------- Fig 1: ECE across the shift ladder (six seeds) ----------
L = pd.read_csv(v2_find('ladder_rows*.csv')); E = pd.read_csv(v2_find('ladder_ens*.csv'))
cfgs = [('B0', 'ce'), ('B0', 'focal'), ('B3', 'ce'), ('B3', 'focal')]
meths = [('raw', 'Raw', OI['grey'], ''), ('cleanTS', 'Clean-fit TS', OI['blue'], '///'), ('ens_TS', 'Six-seed ensemble + TS', OI['green'], '...'), ('oracle100', 'Target-fit TS (100 labels)', OI['verm'], 'xxx')]
fig, axs = plt.subplots(2, 2, figsize=(7.2, 4.4), sharey=True)
for ax, (bb, v) in zip(axs.ravel(), cfgs):
    x = np.arange(4); w = 0.2
    for j, (m, lab, c, h) in enumerate(meths):
        vals, errs = [], []
        for st in SETS:
            d = E[(E.bb == bb) & (E.v == v) & (E.set == st)].ece if m == 'ens_TS' else L[(L.bb == bb) & (L.v == v) & (L.set == st) & (L.method == m)].ece
            vals.append(d.mean() if len(d) else np.nan); errs.append(d.std() if (len(d) > 1 and m != 'ens_TS') else 0)
        ax.bar(x + (j - 1.5) * w, vals, w, yerr=[np.minimum(errs, np.nan_to_num(vals)), errs], color=c, hatch=h, edgecolor='white', linewidth=0.4, error_kw=dict(lw=0.6, capsize=1.5), label=lab)
    ax.set_xticks(x); ax.set_xticklabels([SN[s] for s in SETS], fontsize=7); ax.set_title(f'EfficientNet-{bb}, ' + ('cross-entropy' if v == 'ce' else 'focal loss'), fontsize=8)
for a in axs[:, 0]: a.set_ylabel('ECE (15 bins)')
h, l = axs[0, 0].get_legend_handles_labels(); fig.legend(h, l, loc='upper center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.04)); fig.tight_layout(); v2_save(fig, 'fig1_ece_ladder')

# ---------- Fig 2: reliability diagrams, PAD-UFES-20 (EfficientNet-B0, cross-entropy, seed 0) ----------
def v2_fitT(z, y):
    f = lambda t: -np.mean(log_softmax(z / t, 1)[np.arange(len(y)), y]); return minimize_scalar(f, bounds=(0.05, 10), method='bounded').x
def v2_ece(conf, acc, nb=15):
    b = (np.ceil(conf * nb) - 1).clip(0, nb - 1).astype(int); e = 0.0
    for k in range(nb):
        m = b == k
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e
d = np.load(v2_find('preds_ce_s0.npz')); zc, yc, zt, yt = d['z_cal'], d['y_cal'], d['z_ext_pad'], d['y_ext_pad']
Tc = v2_fitT(zc, yc); rs = np.random.RandomState(0); idx = rs.permutation(len(yt)); a, b_ = idx[:100], idx[100:]; To = v2_fitT(zt[a], yt[a]); yb = yt[b_]
panels = [('Raw', softmax(zt[b_], 1)), (f'Clean-fit TS (T = {Tc:.2f})', softmax(zt[b_] / Tc, 1)), (f'Target-fit TS (T = {To:.2f})', softmax(zt[b_] / To, 1))]
fig, axs = plt.subplots(2, 3, figsize=(7.2, 3.6), gridspec_kw=dict(height_ratios=[3, 1], hspace=0.08), sharex=True); bins = np.linspace(0, 1, 11)
for j, (ttl, p) in enumerate(panels):
    conf, acc = p.max(1), (p.argmax(1) == yb).astype(float); ax = axs[0, j]; ax.plot([0, 1], [0, 1], color='k', lw=0.6, ls='--')
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() >= 5: ax.bar((lo + hi) / 2, acc[m].mean(), hi - lo, color=OI['blue'], edgecolor='white', linewidth=0.4, alpha=0.9); ax.plot([(lo + hi) / 2], [conf[m].mean()], marker='o', ms=2.5, color=OI['verm'])
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_title(f'{ttl}\nECE = {v2_ece(conf, acc):.3f}', fontsize=7.5)
    if j == 0: ax.set_ylabel('Accuracy (bar) / mean confidence (dot)')
    else: ax.set_yticklabels([])
    axs[1, j].hist(conf, bins=bins, color=OI['grey'], edgecolor='white', linewidth=0.4); axs[1, j].set_xlabel('Confidence'); axs[1, j].set_yticks([])
axs[1, 0].set_ylabel('Count'); v2_save(fig, 'fig2_reliability_pad')

# ---------- Fig 3: matcher picks, 12 exploratory and 12 confirmatory models ----------
m1 = pd.read_csv(v2_find('m1_rows*.csv')); m3 = pd.read_csv(v2_find('m3_rows*.csv')); M1 = m1[m1.method == 'matched']
kinds = ['clean', 'blur', 'jpeg', 'color', 'noise', 'lowres', 'combo']
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.1), sharey=True)
for ax, (ttl, dd) in zip(axs, [('Exploratory, seeds 0-2', M1), ('Confirmatory, seeds 3-5', m3)]):
    cnt = np.array([[(dd[dd.set == st].pick == k).sum() for k in kinds] for st in SETS]); ax.imshow(cnt, cmap='Blues', vmin=0, vmax=12, aspect='auto')
    ax.set_xticks(range(7)); ax.set_xticklabels(kinds, rotation=30, ha='right', fontsize=7); ax.set_yticks(range(4)); ax.set_yticklabels([SN[s] for s in SETS], fontsize=7)
    for i in range(4):
        for j in range(7): ax.text(j, i, cnt[i, j], ha='center', va='center', fontsize=7, color='white' if cnt[i, j] > 6 else 'black')
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_title(ttl + ' (12 models)', fontsize=7.5)
fig.suptitle('Corruption chosen by the label-free matcher', fontsize=8, y=1.03); fig.tight_layout(); v2_save(fig, 'fig3_matcher_picks')

# ---------- Fig 4: change in ECE, matched minus clean-fit, both cohorts ----------
P = m1[m1.method.isin(['cleanTS', 'matched'])].pivot_table(index=['bb', 'v', 'seed', 'set'], columns='method', values='ece').reset_index()
P = P.merge(M1[['bb', 'v', 'seed', 'set', 'pick']], on=['bb', 'v', 'seed', 'set']); P['delta'] = P.matched - P.cleanTS; P['cohort'] = 'exploratory'
Q = m3.copy(); Q['cohort'] = 'confirmatory'
if 'bb' not in Q.columns: Q['bb'] = 'B0'
cols = ['bb', 'set', 'pick', 'delta', 'cohort']; A = pd.concat([P[cols], Q[cols]], ignore_index=True)
pc = dict(noise=OI['verm'], lowres=OI['blue'], jpeg=OI['green'], blur=OI['orange'], combo=OI['purple'], color=OI['sky'], clean=OI['grey'])
fig, ax = plt.subplots(figsize=(5.2, 2.9)); seen = set(); xt = []; xl = []
for xi, (st, co) in enumerate([(s, c) for s in ['ext_pad', 'ext_ham'] for c in ['exploratory', 'confirmatory']]):
    dd = A[(A.set == st) & (A.cohort == co)].reset_index(drop=True); jit = np.linspace(-0.22, 0.22, max(len(dd), 2))
    for k, r in dd.iterrows():
        lab = r.pick if r.pick not in seen else None; seen.add(r.pick)
        ax.scatter(xi + jit[k], r.delta, color=pc[r.pick], marker='o' if r.bb == 'B0' else '^', s=20, edgecolor='k', linewidth=0.3, label=lab, zorder=3)
    if len(dd): ax.hlines(dd.delta.mean(), xi - 0.3, xi + 0.3, color='k', lw=1.0, zorder=4)
    xt.append(xi); xl.append(f'{SN[st]}\n{co}')
ax.axhline(0, color='k', lw=0.6); ax.set_xticks(xt); ax.set_xticklabels(xl, fontsize=6.5); ax.set_ylabel('Change in ECE: matched minus clean-fit\n(below 0 = matched better)')
ax.legend(title='matcher pick\n(circle B0, triangle B3;\nbar = cohort mean)', frameon=False, fontsize=6.5, title_fontsize=6.5, loc='center left', bbox_to_anchor=(1.01, 0.5)); fig.tight_layout(); v2_save(fig, 'fig4_matched_vs_clean')

# ---------- Fig 5: Grad-CAM faithfulness against random AND centre order, per model ----------
Fa = pd.DataFrame([json.loads(l) for l in open(v2_find('faith_all_models*.jsonl'))]); Ce = pd.DataFrame([json.loads(l) for l in open(v2_find('*centre*.jsonl'))])
Mg = Fa.merge(Ce, on=['tag', 'set', 'i']); tests = [('ins', 'g', 'Insertion\ngrey'), ('del', 'b', 'Deletion\nblur'), ('ins', 'b', 'Insertion\nblur'), ('del', 'g', 'Deletion\ngrey')]
fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.5), sharey=True); rg = np.random.RandomState(1)
for ax, (st, nm) in zip(axs, [('test_id', 'In-domain'), ('ext_pad', 'PAD-UFES-20'), ('ext_ham', 'HAM10000')]):
    g = Mg[Mg.set == st]
    for ti, (dr, ft, lab) in enumerate(tests):
        for ci, (oth, col) in enumerate([('rnd', OI['grey']), ('cen', OI['blue'])]):
            ad = g.groupby('tag').apply(lambda q: (q[f'cam_{dr}_{ft}'] - q[f'{oth}_{dr}_{ft}']).mean() * (1 if dr == 'ins' else -1), include_groups=False) if 'include_groups' in pd.DataFrame.groupby.__code__.co_varnames else g.groupby('tag').apply(lambda q: (q[f'cam_{dr}_{ft}'] - q[f'{oth}_{dr}_{ft}']).mean() * (1 if dr == 'ins' else -1))
            xx = ti + (ci - 0.5) * 0.36
            ax.scatter(xx + rg.uniform(-0.07, 0.07, len(ad)), ad.values, s=6, color=col, alpha=0.7, linewidth=0, zorder=3); ax.hlines(ad.mean(), xx - 0.15, xx + 0.15, color='k', lw=1.0, zorder=4)
    ax.axhline(0, color='k', lw=0.6); ax.set_xticks(range(4)); ax.set_xticklabels([t[2] for t in tests], fontsize=6.5); ax.set_title(nm, fontsize=7.5)
axs[0].set_ylabel('AUC advantage of Grad-CAM\n(above 0 = Grad-CAM more faithful)')
from matplotlib.lines import Line2D
fig.legend(handles=[Line2D([], [], marker='o', ls='', color=OI['grey'], label='vs random order'), Line2D([], [], marker='o', ls='', color=OI['blue'], label='vs centre order'), Line2D([], [], color='k', lw=1.0, label='mean of 24 models')], loc='upper center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.08))
fig.tight_layout(); v2_save(fig, 'fig5_faithfulness')
mine = [f for f in sorted(os.listdir(V2_OUT)) if f[:4] in ('fig1', 'fig2', 'fig3', 'fig4', 'fig5')]
with zipfile.ZipFile('/kaggle/working/figs_final_v.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in mine: z.write(os.path.join(V2_OUT, f), f)
print('files:', mine, '| zip MB:', round(os.path.getsize('/kaggle/working/figs_final_v.zip') / 1e6, 2))

In [ ]:
# ---- Cell Q2: final Fig 6 (examples) and Fig 7 (curves incl. centre order), written to NEW folder /kaggle/working/figs_final, GPU ----
import os, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, torchvision
import torchvision.transforms as T, torchvision.transforms.functional as TF
from PIL import Image
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from scipy.special import log_softmax, softmax

q1_BB, q1_VAR, q1_SEED = 'B0', 'ce', 0
q1_NAMES = ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'SCCKA', 'VASC']
q1_SETS = [('test_id', 'In-domain'), ('ext_ham', 'HAM10000'), ('ext_pad', 'PAD-UFES-20')]
q1_NCURVE = 100
q1_OUT = '/kaggle/working/figs_final'; os.makedirs(q1_OUT, exist_ok=True)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
q1_DIRS = ['/kaggle/working', '/kaggle/input/datasets/cvaravinda/calib-results-fresh', '/kaggle/input/datasets/cvaravinda/calib-results-b3', '/kaggle/input/datasets/cvaravinda/calib-results-all2']
def q1_find(name):
    for d in q1_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
plt.rcParams.update({'font.family': 'serif', 'font.size': 7, 'axes.spines.top': False, 'axes.spines.right': False, 'savefig.dpi': 300, 'pdf.fonttype': 42})
pre = '' if q1_BB == 'B0' else 'b3_'
q1_df = pd.read_csv(q1_find('final_splits.csv')); q1_MAP = {l: i for i, l in enumerate([0, 1, 3, 4, 7, 8, 9, 10])}
q1_mu = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); q1_sd = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
q1_rs = T.Resize((224, 224)); q1_trap = getattr(np, 'trapezoid', None) or np.trapz
if q1_BB == 'B0': q1_m = torchvision.models.efficientnet_b0(weights=None); q1_m.classifier[1] = nn.Linear(1280, 8)
else:             q1_m = torchvision.models.efficientnet_b3(weights=None); q1_m.classifier[1] = nn.Linear(1536, 8)
q1_m.load_state_dict(torch.load(q1_find(f'model_{pre}{q1_VAR}_s{q1_SEED}.pt'), map_location='cpu')); q1_m = q1_m.to(dev).eval()
q1_d = np.load(q1_find(f'preds_{pre}{q1_VAR}_s{q1_SEED}.npz'))
q1_zc, q1_yc = q1_d['z_cal'], q1_d['y_cal']
q1_Tc = minimize_scalar(lambda t: -np.mean(log_softmax(q1_zc / t, 1)[np.arange(len(q1_yc)), q1_yc]), bounds=(0.05, 10), method='bounded').x
print(f'model {q1_BB} {q1_VAR} seed {q1_SEED}; clean-fit temperature T = {q1_Tc:.3f}; device {dev}')

def q1_img(p):
    im = q1_rs(Image.open(p).convert('RGB')); a = np.asarray(im).copy()
    x = (torch.from_numpy(a).permute(2, 0, 1).float().div(255).unsqueeze(0).to(dev) - q1_mu) / q1_sd
    return a, x
def q1_cam(x):
    with torch.enable_grad():
        a = q1_m.features(x); a.retain_grad(); out = q1_m.classifier(torch.flatten(q1_m.avgpool(a), 1))
        c = int(out.argmax(1)); q1_m.zero_grad(); out[0, c].backward()
        cam = F.relu((a.grad.mean((2, 3), keepdim=True) * a).sum(1, keepdim=True)); cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)[0, 0]
    return cam.detach(), c, out.detach().cpu().numpy()[0]
def q1_curves(x, score, c, base):
    flat = score.flatten(); rank = torch.empty_like(flat); rank[flat.argsort(descending=True)] = torch.arange(flat.numel(), device=dev, dtype=flat.dtype); rank = rank.view(1, 1, 224, 224)
    st = np.linspace(0, 1, 21); res = {}
    with torch.no_grad():
        for tag in ('del', 'ins'):
            xs = []
            for f in st:
                mk = (rank < f * 224 * 224).float(); xs.append(base * mk + x * (1 - mk) if tag == 'del' else x * mk + base * (1 - mk))
            res[tag] = F.softmax(q1_m(torch.cat(xs)), 1)[:, c].cpu().numpy()
    return res

# ---- Figure 6 ----
rows = []
for st, nm in q1_SETS:
    g = q1_df[q1_df.set == st].sample(60, random_state=11); got = {}
    for _, r in g.iterrows():
        y = q1_MAP.get(int(r['label']), -1)
        if y < 0: continue
        a, x = q1_img(r['path']); cam, c, z = q1_cam(x); ok = int(c == y)
        if ok not in got: got[ok] = (nm, a, cam.cpu().numpy(), c, y, z)
        if len(got) == 2: break
    for ok in (1, 0):
        if ok in got: rows.append((ok,) + got[ok])
fig, axs = plt.subplots(len(rows), 3, figsize=(6.4, 1.35 * len(rows)), gridspec_kw=dict(width_ratios=[1, 1, 1.7], wspace=0.12))
for i, (ok, nm, a, cam, c, y, z) in enumerate(rows):
    pr, pt = softmax(z), softmax(z / q1_Tc)
    axs[i, 0].imshow(a); axs[i, 0].set_ylabel(f'{nm}\n' + ('correct' if ok else 'wrong'), fontsize=7)
    axs[i, 1].imshow(a); axs[i, 1].imshow(cam / (cam.max() + 1e-9), cmap='jet', alpha=0.45)
    for j in (0, 1): axs[i, j].set_xticks([]); axs[i, j].set_yticks([]); [s_.set_visible(False) for s_ in axs[i, j].spines.values()]
    axs[i, 0].set_title(f'true: {q1_NAMES[y]}', fontsize=6.5); axs[i, 1].set_title(f'predicted: {q1_NAMES[c]}', fontsize=6.5)
    ax = axs[i, 2]; w = 0.35
    if ok:
        ax.bar([-w / 2], [pr[c]], w, color='#8c8c8c', label='raw'); ax.bar([w / 2], [pt[c]], w, color='#0072B2', label='after TS'); ax.set_xticks([0]); ax.set_xticklabels(['predicted = true class'], fontsize=6.5)
    else:
        xs = np.arange(2); ax.bar(xs - w / 2, [pr[c], pr[y]], w, color='#8c8c8c', label='raw'); ax.bar(xs + w / 2, [pt[c], pt[y]], w, color='#0072B2', label='after TS')
        ax.set_xticks(xs); ax.set_xticklabels(['predicted class', 'true class'], fontsize=6.5)
    ax.set_xlim(-0.7, 1.7); ax.set_ylim(0, 1); ax.tick_params(labelsize=6.5); ax.set_ylabel('probability', fontsize=6.5)
    if i == 0: ax.legend(frameon=False, fontsize=6.5, loc='upper right')
fig.suptitle(f'EfficientNet-{q1_BB[1:]}, {"cross-entropy" if q1_VAR == "ce" else "focal"}, seed {q1_SEED}: first correct and first wrong prediction per set (fixed random order)', fontsize=7.2, y=1.0)
fig.savefig(f'{q1_OUT}/fig6_examples.pdf', bbox_inches='tight', dpi=200); fig.savefig(f'{q1_OUT}/fig6_examples.png', bbox_inches='tight', dpi=200); plt.close(fig); print('saved fig6_examples')

# ---- Figure 7 ----
q1_yy, q1_xx = torch.meshgrid(torch.arange(224, device=dev).float(), torch.arange(224, device=dev).float(), indexing='ij'); q1_cen = -((q1_yy - 111.5) ** 2 + (q1_xx - 111.5) ** 2)
st_x = np.linspace(0, 1, 21); fig, axs = plt.subplots(2, 3, figsize=(7.2, 3.9), sharex=False, sharey=True); auc = {}
sty = {'cam': ('#0072B2', '-', 'Grad-CAM'), 'cen': ('#D55E00', '--', 'centre order'), 'rnd': ('#8c8c8c', ':', 'random order')}
for k, (st, nm) in enumerate(q1_SETS):
    g = q1_df[q1_df.set == st].sample(q1_NCURVE, random_state=7); C = {(s_, t_): [] for s_ in sty for t_ in ('del', 'ins')}
    for i, (_, r) in enumerate(g.iterrows()):
        a, x = q1_img(r['path']); cam, c, _ = q1_cam(x); base = TF.gaussian_blur(x, kernel_size=[51, 51], sigma=[15.0, 15.0])
        rnd = torch.rand(224, 224, device=dev, generator=torch.Generator(device=dev).manual_seed(1000 + i))
        for src, sc in (('cam', cam), ('cen', q1_cen), ('rnd', rnd)):
            res = q1_curves(x, sc, c, base)
            for tg in ('del', 'ins'): C[(src, tg)].append(res[tg])
    for j, tg in enumerate(('del', 'ins')):
        ax = axs[j, k]
        for src in ('rnd', 'cen', 'cam'):
            col, ls, lab = sty[src]; A = np.array(C[(src, tg)]); m = A.mean(0); se = A.std(0, ddof=1) / np.sqrt(len(A))
            ax.plot(st_x, m, ls, color=col, lw=1.2, label=lab); ax.fill_between(st_x, m - 1.96 * se, m + 1.96 * se, color=col, alpha=0.12, lw=0)
            auc[(nm, tg, src)] = q1_trap(m, st_x)
        if j == 0: ax.set_title(nm, fontsize=7.5)
        if k == 0: ax.set_ylabel(('Deletion' if tg == 'del' else 'Insertion') + '\nprobability of predicted class', fontsize=7)
        ax.set_xlabel('Fraction of pixels removed' if tg == 'del' else 'Fraction of pixels revealed', fontsize=7)
axs[0, 0].legend(frameon=False, fontsize=6.5, loc='upper right')
fig.tight_layout(); fig.savefig(f'{q1_OUT}/fig7_curves.pdf', bbox_inches='tight'); fig.savefig(f'{q1_OUT}/fig7_curves.png', bbox_inches='tight', dpi=250); plt.close(fig); print('saved fig7_curves')
for nm in [n for _, n in q1_SETS]:
    for tg in ('del', 'ins'): print(f'{nm:12s} {tg}: AUC of mean curve  Grad-CAM {auc[(nm, tg, "cam")]:.3f} | centre {auc[(nm, tg, "cen")]:.3f} | random {auc[(nm, tg, "rnd")]:.3f}')
import zipfile
with zipfile.ZipFile('/kaggle/working/figs_final_q.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ('fig6_examples.pdf', 'fig6_examples.png', 'fig7_curves.pdf', 'fig7_curves.png'): z.write(f'{q1_OUT}/{f}', f)
print('zip:', round(os.path.getsize('/kaggle/working/figs_final_q.zip') / 1e6, 2), 'MB')

In [ ]:
# ---- Cell Q3: Fig 6 only, layout fixed (no overlapping labels, no suptitle), written to /kaggle/working/figs_final ----
import os, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, torchvision
import torchvision.transforms as T, torchvision.transforms.functional as TF
from PIL import Image
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from scipy.special import log_softmax, softmax

q1_BB, q1_VAR, q1_SEED = 'B0', 'ce', 0
q1_NAMES = ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'SCCKA', 'VASC']
q1_SETS = [('test_id', 'In-domain'), ('ext_ham', 'HAM10000'), ('ext_pad', 'PAD-UFES-20')]
q1_OUT = '/kaggle/working/figs_final'; os.makedirs(q1_OUT, exist_ok=True)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
q1_DIRS = ['/kaggle/working', '/kaggle/input/datasets/cvaravinda/calib-results-fresh', '/kaggle/input/datasets/cvaravinda/calib-results-b3', '/kaggle/input/datasets/cvaravinda/calib-results-all2']
def q1_find(name):
    for d in q1_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
plt.rcParams.update({'font.family': 'serif', 'font.size': 7, 'axes.spines.top': False, 'axes.spines.right': False, 'savefig.dpi': 300, 'pdf.fonttype': 42})
pre = '' if q1_BB == 'B0' else 'b3_'
q1_df = pd.read_csv(q1_find('final_splits.csv')); q1_MAP = {l: i for i, l in enumerate([0, 1, 3, 4, 7, 8, 9, 10])}
q1_mu = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); q1_sd = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
q1_rs = T.Resize((224, 224))
if q1_BB == 'B0': q1_m = torchvision.models.efficientnet_b0(weights=None); q1_m.classifier[1] = nn.Linear(1280, 8)
else:             q1_m = torchvision.models.efficientnet_b3(weights=None); q1_m.classifier[1] = nn.Linear(1536, 8)
q1_m.load_state_dict(torch.load(q1_find(f'model_{pre}{q1_VAR}_s{q1_SEED}.pt'), map_location='cpu')); q1_m = q1_m.to(dev).eval()
q1_d = np.load(q1_find(f'preds_{pre}{q1_VAR}_s{q1_SEED}.npz'))
q1_zc, q1_yc = q1_d['z_cal'], q1_d['y_cal']
q1_Tc = minimize_scalar(lambda t: -np.mean(log_softmax(q1_zc / t, 1)[np.arange(len(q1_yc)), q1_yc]), bounds=(0.05, 10), method='bounded').x
print(f'model {q1_BB} {q1_VAR} seed {q1_SEED}; clean-fit temperature T = {q1_Tc:.3f}; device {dev}')

def q1_img(p):
    im = q1_rs(Image.open(p).convert('RGB')); a = np.asarray(im).copy()
    x = (torch.from_numpy(a).permute(2, 0, 1).float().div(255).unsqueeze(0).to(dev) - q1_mu) / q1_sd
    return a, x
def q1_cam(x):
    with torch.enable_grad():
        a = q1_m.features(x); a.retain_grad(); out = q1_m.classifier(torch.flatten(q1_m.avgpool(a), 1))
        c = int(out.argmax(1)); q1_m.zero_grad(); out[0, c].backward()
        cam = F.relu((a.grad.mean((2, 3), keepdim=True) * a).sum(1, keepdim=True)); cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)[0, 0]
    return cam.detach(), c, out.detach().cpu().numpy()[0]

rows = []
for st, nm in q1_SETS:
    g = q1_df[q1_df.set == st].sample(60, random_state=11); got = {}
    for _, r in g.iterrows():
        y = q1_MAP.get(int(r['label']), -1)
        if y < 0: continue
        a, x = q1_img(r['path']); cam, c, z = q1_cam(x); ok = int(c == y)
        if ok not in got: got[ok] = (nm, a, cam.cpu().numpy(), c, y, z)
        if len(got) == 2: break
    for ok in (1, 0):
        if ok in got: rows.append((ok,) + got[ok])
fig, axs = plt.subplots(len(rows), 3, figsize=(6.4, 1.75 * len(rows)), gridspec_kw=dict(width_ratios=[1, 1, 1.7], wspace=0.3, hspace=0.62))
for i, (ok, nm, a, cam, c, y, z) in enumerate(rows):
    pr, pt = softmax(z), softmax(z / q1_Tc)
    axs[i, 0].imshow(a); axs[i, 0].set_ylabel(f'{nm}\n' + ('correct' if ok else 'wrong'), fontsize=7)
    axs[i, 1].imshow(a); axs[i, 1].imshow(cam / (cam.max() + 1e-9), cmap='jet', alpha=0.45)
    for j in (0, 1): axs[i, j].set_xticks([]); axs[i, j].set_yticks([]); [s_.set_visible(False) for s_ in axs[i, j].spines.values()]
    axs[i, 0].set_title(f'true: {q1_NAMES[y]}', fontsize=6.5, pad=3); axs[i, 1].set_title(f'predicted: {q1_NAMES[c]}', fontsize=6.5, pad=3)
    ax = axs[i, 2]; w = 0.35
    if ok:
        ax.bar([-w / 2], [pr[c]], w, color='#8c8c8c', label='raw'); ax.bar([w / 2], [pt[c]], w, color='#0072B2', label='after TS'); ax.set_xticks([0]); ax.set_xticklabels(['predicted = true class'], fontsize=6.5)
    else:
        xs = np.arange(2); ax.bar(xs - w / 2, [pr[c], pr[y]], w, color='#8c8c8c', label='raw'); ax.bar(xs + w / 2, [pt[c], pt[y]], w, color='#0072B2', label='after TS')
        ax.set_xticks(xs); ax.set_xticklabels(['predicted class', 'true class'], fontsize=6.5)
    ax.set_xlim(-0.7, 1.7); ax.set_ylim(0, 1); ax.tick_params(labelsize=6.5); ax.set_ylabel('probability', fontsize=6.5)
    if i == 0: ax.legend(frameon=False, fontsize=6.5, loc='upper right')
fig.savefig(f'{q1_OUT}/fig6_examples.pdf', bbox_inches='tight', dpi=200); fig.savefig(f'{q1_OUT}/fig6_examples.png', bbox_inches='tight', dpi=200); plt.close(fig); print('saved fig6_examples')
import zipfile
with zipfile.ZipFile('/kaggle/working/figs_final_q3.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ('fig6_examples.pdf', 'fig6_examples1.png'): z.write(f'{q1_OUT}/{f}', f)
print('zip:', round(os.path.getsize('/kaggle/working/figs_final_q3.zip') / 1e6, 2), 'MB')